# NeuralSS data pipeline: discovery, copy detection and leakage-safe splits (Kaggle GPU T4 x2)

**Goal.** Build the single dataset manifest used by both training notebooks of the NeuralSS stack
(temporal super-resolution and low-latency frame generation). Only Kaggle and Hugging Face datasets are used.

**Steps.**

1. Discover attached Kaggle datasets (capped, randomised directory walk; official test lists read directly) and drop degraded copies (LR, bicubic, blurred, `x2`/`x3`/`x4`) and non-colour passes.
2. Materialise selected Hugging Face subsets into `/kaggle/working/nss_data` with keep-alive range requests and single-stream reads (no full archive downloads): game frames, TartanAir sequences with depth, exact optical flow and occlusion masks, GameIR native 720p / 1440p game renders with depth, and 1080p Vimeo videos.
3. Group frames into sequences; every item gets a leakage group (sequence, video, scene family, environment or town).
4. Decode keyframes into thumbnails and compute quality statistics; filter too-small, flat, letterboxed and blurry items.
5. Copy detection across the whole combined dataset in two stages. Candidates: DINOv2-small nearest neighbours (exhaustive GPU search) and pHash pairs. Verification: SIFT matches that agree on one RANSAC affine map, scored by the image area they cover. Thresholds are calibrated on synthetic copies against random and hard negatives (nearest neighbours between items known to be distinct).
6. Split by leakage group: test items keep their role, anything sharing a group with a test item or connected to one by duplicate edges is dropped, and the remaining components go to validation or training.
7. Audit every validation and test keyframe against its nearest training (and validation) keyframes with the same rule; any duplicate found is added as an edge and the split is repeated until the audit is clean.

**Why two stages.** The first run used global similarity only (pHash Hamming 14, DINOv2 cosine 0.75). Those thresholds flag *similar* pictures (two cherry-tree photos, two skylines, frames of one game), not copies. The false edges chained 8,354 items into one component that touched a test item, and 8,150 training items (72 percent) were dropped. Geometric verification separates copies from look-alikes. Splitting by provenance instead of transitive components keeps one copied clip from taking its whole folder with it.

**Outputs** (in `/kaggle/working`, saved as this notebook's output; attach it to the training notebooks with **Add Input > Notebook output**):

```text
nss_data/nss_manifest.parquet   one row per still or sequence: source, frames (URIs), motion / depth / mask URIs, group, split, component, drop_reason
nss_data/<hf subsets>/          materialised Hugging Face data referenced by the manifest
metrics/                        discovery, statistics, calibration, duplicate edges, split and audit reports, dataset card
plots/                          every figure as PNG
logs/notebook.log
outputs.zip                     metrics, plots, logs, code and the manifest (the data itself is already in the Output tab)
```

**Required Kaggle settings.** Accelerator **GPU T4 x2**; **Internet on** (Hugging Face data and DINOv2 weights); optional secret `HF_TOKEN` (higher Hub rate limits). Attach these Kaggle datasets:

| Kaggle dataset | Use |
| :--- | :--- |
| `soumikrakshit/div2k-high-resolution-images` | stills; `DIV2K_valid_HR` pinned to test |
| `daehoyang/flickr2k` | stills |
| `amithkesavmrajagiri/reds-dataset` | 720p video sequences |
| `wangsally/vimeo-90k-7` | 448x256 septuplets |
| `chenshu123/vimeo-triplet` | frame-generation test (clips from the official test list) |
| `artemmmtry/mpi-sintel-dataset` | rendered sequences with exact flow and occlusions (scene families `market`, `cave` pinned to test) |
| `uom200647r/vid4-dataset` | video test only |

**Run time.** The first full run took 2.7 hours, 2.4 of them Hugging Face transfer. Connection reuse and streamed tars should shorten that, and copy detection adds roughly 10 to 20 minutes of CPU verification (not yet measured). `QUICK_RUN = True` shrinks every cap for a short smoke test.

## 1. Library files

The pipeline code is shared with the two training notebooks. The first cell creates `/kaggle/working/code`. The following cells write the library into it so the notebook is self-contained: `nss_common` (run folders, logging, stage timer), `nss_data` (source registry, discovery, Hugging Face access, decoding, windows), `nss_dedup` (hashes, embeddings, GPU neighbour search, SIFT verification, calibration copies, split) and `nss_pipeline` (the steps run below).

In [ ]:
import os

os.makedirs(
    "/kaggle/working/code", exist_ok=True
)  # %%writefile does not create folders

**`nss_common.py`**: run folders, logger, stage timer, figure and metric saving, hardware report, pip helper and the training-process launcher.

In [ ]:
%%writefile /kaggle/working/code/nss_common.py
"""Shared notebook runtime: output folders, logging, stage timing, figure saving, hardware report, pip helper."""

import importlib
import json
import logging
import os
import shutil
import subprocess
import sys
import time
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import psutil
import torch

WORK = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
SUBDIRS = ["code", "weights", "plots", "metrics", "logs", "predictions"]


class Run:
    """Holds the run clock, output folders, logger and per-stage timings of one notebook run."""

    def __init__(self, name, work=WORK):
        self.start = time.time()
        self.work = Path(work)
        self.dirs = {d: self.work / d for d in SUBDIRS}
        for folder in self.dirs.values():
            folder.mkdir(parents=True, exist_ok=True)
        self.log = logging.getLogger(name)
        self.log.setLevel(logging.INFO)
        self.log.handlers.clear()
        self.log.propagate = False
        for handler in (
            logging.StreamHandler(sys.stdout),
            logging.FileHandler(self.dirs["logs"] / "notebook.log"),
        ):
            handler.setFormatter(
                logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")
            )
            self.log.addHandler(handler)
        self.stage_times = {}

    def hours(self):
        return (time.time() - self.start) / 3600

    @contextmanager
    def stage(self, name):
        t0 = time.time()
        self.log.info("start: %s", name)
        try:
            yield
        finally:
            self.stage_times[name] = time.time() - t0
            self.log.info(
                "done: %s in %.1f min (run total %.2f h)",
                name,
                self.stage_times[name] / 60,
                self.hours(),
            )

    def show(self, fig, name):
        from IPython.display import display

        fig.savefig(self.dirs["plots"] / f"{name}.png", dpi=120, bbox_inches="tight")
        display(fig)
        plt.close(fig)

    def save_json(self, obj, name):
        with open(self.dirs["metrics"] / name, "w") as f:
            json.dump(obj, f, indent=2, default=str)

    def save_csv(self, df, name, **kwargs):
        df.to_csv(self.dirs["metrics"] / name, **kwargs)


def hardware_report(work=WORK):
    """GPU, CPU, RAM and disk detected at run time (nothing hard-coded)."""
    gpus = []
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        gpus.append(
            {
                "gpu": i,
                "name": p.name,
                "vram_gb": round(p.total_memory / 2**30, 2),
                "sm": p.multi_processor_count,
                "cc": f"{p.major}.{p.minor}",
            }
        )
    vm = psutil.virtual_memory()
    report = {
        "gpus": len(gpus),
        "gpu_list": gpus,
        "cpu_count": os.cpu_count(),
        "ram_total_gb": round(vm.total / 2**30, 1),
        "ram_available_gb": round(vm.available / 2**30, 1),
        "disk_free_gb_working": round(shutil.disk_usage(work).free / 2**30, 1),
        "disk_free_gb_tmp": (
            round(shutil.disk_usage("/tmp").free / 2**30, 1)
            if Path("/tmp").exists()
            else None
        ),
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
    }
    return report


def pip_install(packages, log=None):
    """Install missing packages quietly; returns {package: importable}. Needs Kaggle Internet on."""
    status = {}
    for pip_name, module in packages.items():
        try:
            importlib.import_module(module)
            status[pip_name] = True
            continue
        except ImportError:
            pass
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            capture_output=True,
            text=True,
        )
        try:
            importlib.invalidate_caches()
            importlib.import_module(module)
            status[pip_name] = True
        except ImportError:
            status[pip_name] = False
            if log:
                log.warning(
                    "could not install %s: %s", pip_name, result.stderr.strip()[-300:]
                )
    return status


def gpu_utilisation():
    try:
        out = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=10,
        ).stdout
        return [
            [float(v) for v in line.split(",")] for line in out.strip().splitlines()
        ]
    except Exception:
        return []


def stream_process(cmd, env, on_line, log_path):
    """Run a subprocess, call on_line for every stdout line, mirror everything into log_path; returns exit code."""
    with open(log_path, "w") as logf:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        for line in proc.stdout:
            logf.write(line)
            logf.flush()
            on_line(line.rstrip("\n"))
        return proc.wait()


def launch_training(
    script, cfg, cfg_path, log_path, gpus, desc, port=29500, on_round=None
):
    """Run a training script on the given GPUs (torch.distributed.run for several GPUs, a plain process for one),
    write its config, mirror its output to log_path, drive a tqdm bar from PROGRESS lines and collect ROUND rows.
    Returns (rounds, done). Raises with the log tail if the process fails."""
    from tqdm.auto import tqdm

    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)
    env = os.environ.copy()
    env.update(
        {
            "NCCL_P2P_DISABLE": "1",
            "OMP_NUM_THREADS": "1",
            "PYTHONUNBUFFERED": "1",
            "CUDA_VISIBLE_DEVICES": ",".join(str(g) for g in gpus),
        }
    )
    if len(gpus) > 1:
        cmd = [
            sys.executable,
            "-m",
            "torch.distributed.run",
            f"--nproc_per_node={len(gpus)}",
            f"--master_port={port}",
            str(script),
            "--config",
            str(cfg_path),
        ]
    else:
        env["CUDA_DEVICE"] = "0"
        cmd = [sys.executable, str(script), "--config", str(cfg_path)]
    rounds, done, tail = [], {}, []
    bar = tqdm(
        total=100,
        desc=desc,
        unit="%",
        bar_format="{l_bar}{bar}| {n:.0f}/{total:.0f}% [{elapsed}<{remaining}] {postfix}",
    )

    def on_line(line):
        tail.append(line)
        del tail[:-40]
        kind, _, payload = line.partition(" ")
        if kind in ("PROGRESS", "ROUND", "DONE", "LOG"):
            try:
                data = json.loads(payload)
            except ValueError:
                return
            if kind == "PROGRESS":
                bar.n = round(100 * data["progress"], 1)
                bar.set_postfix(
                    loss=f"{data['loss']:.4f}",
                    sps=f"{data['samples_per_s']:.0f}",
                    lr=f"{data['lr']:.1e}",
                    mem=data["mem_gb"],
                )
                bar.refresh()
            elif kind == "ROUND":
                rounds.append(data)
                if on_round:
                    on_round(rounds)
            elif kind == "DONE":
                done.update(data)
            else:
                print(f"[{desc}] {data.get('msg')}", flush=True)
        elif "Error" in line or "Traceback" in line:
            print(f"[{desc}] {line}", flush=True)

    code = stream_process(cmd, env, on_line, log_path)
    bar.n = 100
    bar.close()
    if code != 0:
        raise RuntimeError(
            f"{desc} failed with exit code {code}; last lines:\n"
            + "\n".join(tail[-15:])
        )
    return rounds, done

**`nss_data.py`**: source registry (Kaggle and Hugging Face), capped discovery, keep-alive HTTP with range requests and streamed tars, Hugging Face materialisation, decoding of colour / depth / flow / masks, window and crop workers, manifest I/O.

In [ ]:
%%writefile /kaggle/working/code/nss_data.py
"""Unified dataset layer: source registry (Kaggle + Hugging Face), discovery, filtering, materialisation of
Hugging Face subsets, sequence building, decoding of colour / depth / flow / masks, and window extraction.

URI scheme used in the manifest (so it stays valid across notebooks and machines):
  kaggle:<owner/slug>|<path inside the dataset>   resolved under /kaggle/input
  data:<relative path>                            resolved under the data root (materialised Hugging Face subsets)
  mp4:<relative path>#<frame index>               a frame of a video file under the data root
"""

import hashlib
import io
import json
import os
import random
import re
import tarfile
import threading
import time
import urllib.parse
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
from PIL import Image

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff", ".ppm"}
# Path tokens that mark degraded copies or non-colour passes; such files are never used as targets.
LOW_QUALITY_TOKEN = re.compile(
    r"^(lr\d*|lq|low|lowres|bicubic|bix\d|bdx\d|lrbi|lrbd|x\d|blur|blurred|noisy|compressed|jpeg)$"
)
NON_COLOUR_TOKENS = {
    "occlusions",
    "occlusion",
    "invalid",
    "flow",
    "flow_viz",
    "viz",
    "mask",
    "masks",
    "depth",
    "albedo",
    "shading",
    "disparities",
    "segmentation",
    "camdata",
    "camdata_left",
    "edges",
    "normals",
    "seg",
}
HR_TOKENS = {"hr", "gt", "original", "groundtruth"}
HF = "https://huggingface.co"

# ---------------------------------------------------------------------------------------------------------------
# Source registry. Every source states where it lives, what it provides and how it may be used.
# role: train sources are split into train/val by leakage group; test sources (or test parts) are pinned to test.
# distinct_groups: different groups never show the same content (used for hard negatives when calibrating).
# ---------------------------------------------------------------------------------------------------------------
SOURCES = [
    {
        "name": "div2k",
        "host": "kaggle",
        "ref": "soumikrakshit/div2k-high-resolution-images",
        "kind": "still",
        "test_dirs": ["DIV2K_valid_HR"],
        "distinct_groups": True,
        "licence": "DIV2K: academic research use",
        "content": "photo",
    },
    {
        "name": "flickr2k",
        "host": "kaggle",
        "ref": "daehoyang/flickr2k",
        "kind": "still",
        "licence": "Flickr2K: research use (Flickr images, original licences)",
        "content": "photo",
    },
    {
        "name": "reds",
        "host": "kaggle",
        "ref": "amithkesavmrajagiri/reds-dataset",
        "kind": "video",
        "group_level": 1,
        "licence": "REDS: CC BY 4.0",
        "content": "video",
    },
    {
        "name": "vimeo_septuplet",
        "host": "kaggle",
        "ref": "wangsally/vimeo-90k-7",
        "kind": "video",
        "group_level": 2,
        "licence": "Vimeo-90K: research use (Vimeo uploads)",
        "content": "video",
    },
    {
        "name": "vimeo_triplet",
        "host": "kaggle",
        "ref": "chenshu123/vimeo-triplet",
        "kind": "video",
        "group_level": 2,
        "test_list": "tri_testlist.txt",
        "test_only": True,
        "list_cap": 6000,
        "max_items": 1000,
        "licence": "Vimeo-90K: research use",
        "content": "video",
    },
    {
        "name": "sintel",
        "host": "kaggle",
        "ref": "artemmmtry/mpi-sintel-dataset",
        "kind": "render_seq",
        "test_groups": ["market", "cave"],
        "distinct_groups": True,
        "licence": "MPI Sintel: CC BY 3.0 (Kaggle mirror)",
        "content": "rendered",
    },
    {
        "name": "vid4",
        "host": "kaggle",
        "ref": "uom200647r/vid4-dataset",
        "kind": "video",
        "group_level": 1,
        "test_only": True,
        "licence": "Vid4: research benchmark (Kaggle mirror Apache 2.0)",
        "content": "video",
    },
    {
        "name": "game_stills",
        "host": "hf",
        "repo": "ericphann/video-game-super-resolution",
        "kind": "still",
        "train_dir": "training-hr",
        "test_dir": "test-hr",
        "licence": "HF tag Apache 2.0 (game captures; check per use)",
        "content": "game",
    },
    {
        "name": "tartanair",
        "host": "hf",
        "repo": "theairlabcmu/tartanair",
        "kind": "render_seq",
        "train_envs": [
            "abandonedfactory",
            "amusement",
            "carwelding",
            "endofworld",
            "gascola",
            "hospital",
            "neighborhood",
            "oldtown",
            "seasonsforest",
            "soulcity",
            "westerndesert",
            "ocean",
        ],
        "test_envs": ["japanesealley", "seasidetown"],
        "distinct_groups": True,
        "licence": "TartanAir: BSD-3-Clause (HF tag and tools)",
        "content": "rendered",
    },
    {
        "name": "gameir",
        "host": "hf",
        "repo": "LLLebin/GameIR",
        "kind": "render_pairs",
        "train_tars": ["mini_dataset/train/GameIR-SR/GameIR-SR-000000.tar"],
        "test_tars": ["mini_dataset/test/GameIR-SR/GameIR-SR-000000.tar"],
        "licence": "GameIR: MIT",
        "content": "game",
    },
    {
        "name": "vimeo1080p",
        "host": "hf",
        "repo": "danjacobellis/vimeo1080p",
        "kind": "video_file",
        "train_shards": [
            "data/train-00000-of-00045.parquet",
            "data/train-00001-of-00045.parquet",
        ],
        "val_shards": ["data/validation-00000-of-00003.parquet"],
        "licence": "unspecified on HF (Vimeo uploads); research use",
        "content": "video",
    },
]


def source(name):
    return next(s for s in SOURCES if s["name"] == name)


# ---------------------------------------------------------------------------------------------------------------
# Path utilities (Kaggle mounts)
# ---------------------------------------------------------------------------------------------------------------
def tokens(text):
    return [t for t in re.split(r"[_\-\s.]+", text.lower()) if t]


def path_tokens(path, root):
    rel = Path(path).relative_to(root)
    parts = list(rel.parts[:-1]) + [Path(rel.parts[-1]).stem]
    return [t for part in parts for t in tokens(part)]


def is_clean_colour(path, root):
    return not any(
        LOW_QUALITY_TOKEN.match(t) or t in NON_COLOUR_TOKENS
        for t in path_tokens(path, root)
    )


def find_dataset_root(input_root, owner_slug):
    """Mount folder of a Kaggle dataset under either supported layout, or None."""
    owner, slug = owner_slug.split("/")
    input_root = Path(input_root)
    for candidate in (
        input_root / slug,
        input_root / "datasets" / owner / slug,
        input_root / "datasets" / slug,
    ):
        if candidate.is_dir():
            return candidate
    for depth_one in input_root.iterdir() if input_root.is_dir() else []:
        if depth_one.is_dir():
            for depth_two in [depth_one] + [
                p for p in depth_one.iterdir() if p.is_dir()
            ]:
                if depth_two.name == slug:
                    return depth_two
    return None


def sampled_walk(root, max_files, seed, exts=IMG_EXTS):
    """List files, expanding a random directory from the frontier each step, so a capped listing spans many
    sequences. Each directory is listed completely, so a listed sequence folder is never cut in half.
    """
    rng = random.Random(seed)
    frontier, files, dirs_listed = [str(root)], [], 0
    while frontier and len(files) < max_files:
        i = rng.randrange(len(frontier))
        frontier[i], frontier[-1] = frontier[-1], frontier[i]
        current = frontier.pop()
        try:
            entries = list(os.scandir(current))
        except OSError:
            continue
        dirs_listed += 1
        for entry in entries:
            if entry.name.startswith("."):
                continue
            if entry.is_dir(follow_symlinks=False):
                frontier.append(entry.path)
            elif os.path.splitext(entry.name)[1].lower() in exts:
                files.append(entry.path)
    return sorted(files), dirs_listed, len(frontier) == 0


def natural_key(path):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", str(path).lower())]


def stable_hash(text):
    return int(hashlib.md5(text.encode()).hexdigest(), 16)


# ---------------------------------------------------------------------------------------------------------------
# Hugging Face access: plain HTTPS with retries and range requests (no local Hub cache, works for Xet-backed files)
# ---------------------------------------------------------------------------------------------------------------
def _headers():
    token = os.environ.get("HF_TOKEN")
    return {"Authorization": f"Bearer {token}"} if token else {}


def hf_url(repo, path):
    return f"{HF}/datasets/{repo}/resolve/main/{urllib.parse.quote(path)}"


_LOCAL = threading.local()


def _session():
    """One keep-alive HTTP session per thread: range reads reuse the connection instead of a new TLS handshake
    per request. Authorization is dropped on redirects to another host (the signed CDN URL).
    """
    if getattr(_LOCAL, "session", None) is None:
        import requests

        _LOCAL.session = requests.Session()
    return _LOCAL.session


def http_get(url, headers=None, retries=5, timeout=120, expect_len=None):
    """GET with retries and exponential back-off; with expect_len, a short body (truncated transfer) is retried."""
    for attempt in range(retries):
        try:
            r = _session().get(
                url, headers={**_headers(), **(headers or {})}, timeout=timeout
            )
            r.raise_for_status()
            if expect_len is not None and len(r.content) != expect_len:
                raise IOError(f"short read: {len(r.content)} of {expect_len} bytes")
            return r.content, r.headers
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2**attempt)


def http_stream(url, timeout=300):
    """Streaming GET (one connection for a whole sequential read, e.g. a tar); the caller closes the response."""
    r = _session().get(url, headers=_headers(), stream=True, timeout=timeout)
    r.raise_for_status()
    r.raw.decode_content = True
    return r


def http_download(url, dest, retries=5):
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix(dest.suffix + ".part")
    for attempt in range(retries):
        try:
            with http_stream(url) as r, open(tmp, "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
            tmp.replace(dest)
            return dest
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2**attempt)


def hf_list(repo, path=""):
    """All entries under a dataset folder (follows pagination)."""
    url = f"{HF}/api/datasets/{repo}/tree/main/{urllib.parse.quote(path)}"
    out = []
    while url:
        body, headers = http_get(url)
        out.extend(json.loads(body))
        link = headers.get("Link") or ""
        m = re.search(r'<([^>]+)>;\s*rel="next"', link)
        url = m.group(1) if m else None
    return out


class HttpRangeFile(io.RawIOBase):
    """Seekable read-only file over HTTP range requests, so zip members can be read without downloading archives."""

    def __init__(self, url):
        self.source_url, self.pos = url, 0
        self._resolve()

    def _resolve(self):
        """Follow the Hub redirect once; the signed CDN URL is reused until it fails (it expires after a while)."""
        r = _session().head(
            self.source_url, headers=_headers(), allow_redirects=True, timeout=60
        )
        r.raise_for_status()
        self.size = int(r.headers["Content-Length"])
        self.url = r.url

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = {0: offset, 1: self.pos + offset, 2: self.size + offset}[whence]
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        rng = {"Range": f"bytes={self.pos}-{self.pos + n - 1}"}
        try:
            data, _ = http_get(self.url, headers=rng, retries=3, expect_len=n)
        except Exception:
            self._resolve()
            data, _ = http_get(self.url, headers=rng, expect_len=n)
        self.pos += len(data)
        return data

    def readinto(self, b):
        data = self.read(len(b))
        b[: len(data)] = data
        return len(data)


def remote_zip(url):
    return zipfile.ZipFile(io.BufferedReader(HttpRangeFile(url), buffer_size=1 << 20))


# ---------------------------------------------------------------------------------------------------------------
# Decoding
# ---------------------------------------------------------------------------------------------------------------
class Resolver:
    """Maps manifest URIs to local paths."""

    def __init__(self, data_root, input_root="/kaggle/input"):
        self.data_root, self.input_root, self.roots = (
            Path(data_root),
            Path(input_root),
            {},
        )

    def kaggle_root(self, ref):
        if ref not in self.roots:
            self.roots[ref] = find_dataset_root(self.input_root, ref)
        return self.roots[ref]

    def path(self, uri):
        if uri.startswith("kaggle:"):
            ref, rel = uri[7:].split("|", 1)
            root = self.kaggle_root(ref)
            if root is None:
                raise FileNotFoundError(f"Kaggle dataset {ref} is not attached")
            return root / rel
        if uri.startswith("data:"):
            return self.data_root / uri[5:]
        if uri.startswith("mp4:"):
            return self.data_root / uri[4:].split("#")[0]
        raise ValueError(uri)


def kaggle_uri(ref, root, path):
    return f"kaggle:{ref}|{Path(path).relative_to(root).as_posix()}"


def read_video_frames(path, indices):
    import cv2

    cap = cv2.VideoCapture(str(path))
    frames, want = {}, sorted(set(indices))
    if want:
        cap.set(cv2.CAP_PROP_POS_FRAMES, want[0])
        idx = want[0]
        while idx <= want[-1]:
            ok, frame = cap.read()
            if not ok:
                break
            if idx in want:
                frames[idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            idx += 1
    cap.release()
    missing = [i for i in indices if i not in frames]
    if missing:
        raise IOError(f"could not decode frames {missing[:3]} of {path}")
    return [frames[i] for i in indices]


def video_frame_count(path):
    import cv2

    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n


def load_rgb(resolver, uri):
    if uri.startswith("mp4:"):
        return read_video_frames(resolver.path(uri), [int(uri.split("#")[1])])[0]
    with Image.open(resolver.path(uri)) as im:
        return np.asarray(im.convert("RGB"))


def load_rgb_many(resolver, uris):
    """Decode a list of frames, reading each video file once."""
    if (
        uris
        and all(u.startswith("mp4:") for u in uris)
        and len({u.split("#")[0] for u in uris}) == 1
    ):
        return read_video_frames(
            resolver.path(uris[0]), [int(u.split("#")[1]) for u in uris]
        )
    return [load_rgb(resolver, u) for u in uris]


def read_flo(path):
    """Middlebury .flo (Sintel): float32 (H, W, 2)."""
    with open(path, "rb") as f:
        magic = np.frombuffer(f.read(4), np.float32)[0]
        assert abs(magic - 202021.25) < 1e-3, f"bad .flo magic in {path}"
        w, h = np.frombuffer(f.read(8), np.int32)
        return np.frombuffer(f.read(), np.float32).reshape(h, w, 2).copy()


def load_flow(resolver, uri):
    p = resolver.path(uri)
    return read_flo(p) if p.suffix == ".flo" else np.load(p).astype(np.float32)


def load_valid(resolver, uri, kind):
    """Validity of the motion vector of each pixel (1 = the pixel is visible in the previous frame)."""
    p = resolver.path(uri)
    with Image.open(p) as im:
        a = np.asarray(im.convert("L"))
    return (
        (a < 128).astype(np.uint8)
        if kind == "occlusion"
        else (a >= 128).astype(np.uint8)
    )


def decode_carla_depth(rgba):
    """CARLA depth PNG (R + 256 G + 65536 B) / (2^24 - 1) * 1000 m."""
    rgb = rgba[..., :3].astype(np.float64)
    norm = (rgb[..., 0] + rgb[..., 1] * 256.0 + rgb[..., 2] * 65536.0) / (256.0**3 - 1)
    return (norm * 1000.0).astype(np.float32)


def load_depth(resolver, uri):
    p = resolver.path(uri)
    if p.suffix == ".npy":
        return np.load(p).astype(np.float32)
    with Image.open(p) as im:
        return decode_carla_depth(np.asarray(im.convert("RGBA")))


def normalise_depth(d):
    """Unit-free inverse-depth style encoding in [0, 1] (1 = near): 1 / (1 + d / median)."""
    d = np.where(np.isfinite(d) & (d > 0), d, np.nan)
    med = np.nanmedian(d) if np.isfinite(d).any() else 1.0
    out = 1.0 / (1.0 + np.nan_to_num(d, nan=1e9) / max(med, 1e-6))
    return out.astype(np.float32)


# ---------------------------------------------------------------------------------------------------------------
# Catalogue builders: each returns a list of item dicts (one per still image or per sequence)
# ---------------------------------------------------------------------------------------------------------------
def _item(src, kind, group, frames, role, **extra):
    item_id = (
        f"{src['name']}/{group}/{hashlib.md5(frames[0].encode()).hexdigest()[:10]}"
    )
    return {
        "item_id": item_id,
        "source": src["name"],
        "kind": kind,
        "group": f"{src['name']}:{group}",
        "role": role,
        "frames": frames,
        "n_frames": len(frames),
        "content": src.get("content", ""),
        **extra,
    }


def find_shallow(root, name, depth=4):
    """First file called name at most depth levels below root (avoids a recursive walk of a large dataset)."""
    for d in range(depth):
        hits = sorted(Path(root).glob("/".join(["*"] * d + [name])))
        if hits:
            return hits[0]
    return None


def test_list_files(src, root, seed):
    """Frames of up to max_items clips named in the source's official test list (sampled with the seed), read
    directly from <list folder>/sequences/<clip>. Returns (files, dirs listed, True) or None if unavailable.
    """
    lst = find_shallow(root, src["test_list"])
    seq_root = None if lst is None else lst.parent / "sequences"
    if seq_root is None or not seq_root.is_dir():
        return None
    clips = sorted({s.strip() for s in lst.read_text().splitlines() if s.strip()})
    random.Random(seed).shuffle(clips)
    files, dirs = [], 0
    for clip in clips:
        if dirs >= src.get("max_items", len(clips)):
            break
        folder = seq_root / clip
        if folder.is_dir():
            files += [str(f) for f in folder.iterdir() if f.suffix.lower() in IMG_EXTS]
            dirs += 1
    return (sorted(files), dirs, True) if files else None


def catalogue_kaggle(src, input_root, cap, seed):
    """Stills and frame folders of a Kaggle-mounted source, with the degraded-copy filter."""
    root = find_dataset_root(input_root, src["ref"])
    if root is None:
        return [], {"found": False}
    listed = test_list_files(src, root, seed) if src.get("test_list") else None
    files, dirs_listed, complete = listed or sampled_walk(
        root, src.get("list_cap", cap), seed
    )
    kept = [f for f in files if is_clean_colour(f, root)]
    if src["kind"] in ("video",) and any(
        any(t in HR_TOKENS for t in path_tokens(f, root)) for f in kept
    ):
        kept = [f for f in kept if any(t in HR_TOKENS for t in path_tokens(f, root))]
    items = []
    if src["kind"] == "still":
        for f in kept:
            rel = Path(f).relative_to(root)
            role = (
                "test"
                if any(d in rel.parts for d in src.get("test_dirs", []))
                else "train"
            )
            items.append(
                _item(
                    src, "still", Path(f).stem, [kaggle_uri(src["ref"], root, f)], role
                )
            )
    elif src["kind"] == "video":
        test_set = set()
        if src.get("test_list"):
            lst = find_shallow(root, src["test_list"])
            if lst is not None:
                test_set = {
                    line.strip()
                    for line in lst.read_text().splitlines()
                    if line.strip()
                }
        folders = {}
        for f in kept:
            folders.setdefault(str(Path(f).parent), []).append(f)
        for folder, frames in folders.items():
            frames = sorted(frames, key=natural_key)
            if len(frames) < 3:
                continue
            p = Path(folder)
            if p.name.lower() in HR_TOKENS:
                p = p.parent
            level = src.get("group_level", 1)
            group = p.name if level == 1 else p.parent.name
            clip_key = f"{p.parent.name}/{p.name}"
            if src.get("test_only"):
                if test_set and clip_key not in test_set:
                    continue
                role = "test"
            else:
                role = "train"
            items.append(
                _item(
                    src,
                    "seq",
                    group,
                    [kaggle_uri(src["ref"], root, f) for f in frames],
                    role,
                    mv="none",
                )
            )
    elif src["name"] == "sintel":
        items = catalogue_sintel(src, root)
    if src.get("max_items") and len(items) > src["max_items"]:
        items = random.Random(seed).sample(items, src["max_items"])
    return items, {
        "found": True,
        "listed": len(files),
        "kept_files": len(kept),
        "dirs_listed": dirs_listed,
        "listing_complete": complete,
    }


def catalogue_sintel(src, root):
    """Sintel training scenes with exact forward flow and occlusion masks. The leakage group is the scene prefix
    (alley, ambush, market, ...) because numbered scenes share sets and characters."""
    items = []
    for pass_dir in sorted(Path(root).rglob("training/clean")) + sorted(
        Path(root).rglob("training/final")
    ):
        train_root = pass_dir.parent
        for scene in sorted(p for p in pass_dir.iterdir() if p.is_dir()):
            frames = sorted(scene.glob("*.png"), key=natural_key)
            flows = [
                train_root / "flow" / scene.name / (f.stem + ".flo") for f in frames
            ]
            occ = [train_root / "occlusions" / scene.name / f.name for f in frames]
            if len(frames) < 4 or not flows[0].exists():
                continue
            prefix = scene.name.split("_")[0]
            role = "test" if prefix in src.get("test_groups", []) else "train"
            n = len(frames) - 1  # the last frame has no forward flow
            items.append(
                _item(
                    src,
                    "seq",
                    prefix,
                    [kaggle_uri(src["ref"], root, f) for f in frames[:n]],
                    role,
                    mv="exact_forward",
                    flows=[kaggle_uri(src["ref"], root, f) for f in flows[:n]],
                    valid=[kaggle_uri(src["ref"], root, f) for f in occ[:n]],
                    valid_kind="occlusion",
                    variant=pass_dir.name,
                )
            )
    return items


# ---------------------------------------------------------------------------------------------------------------
# Hugging Face materialisation (download only what is needed into the data root)
# ---------------------------------------------------------------------------------------------------------------
def materialise_game_stills(src, data_root, n_train, n_test, seed, log=print):
    rng = random.Random(seed)
    items = []
    for role, folder, n in (
        ("train", src["train_dir"], n_train),
        ("test", src["test_dir"], n_test),
    ):
        names = sorted(
            e["path"]
            for e in hf_list(src["repo"], folder)
            if e["type"] == "file" and e["path"].endswith(".png")
        )
        chosen = sorted(rng.sample(names, min(n, len(names))))
        dests = [Path(data_root) / "game_stills" / p for p in chosen]
        with ThreadPoolExecutor(8) as pool:
            list(
                pool.map(
                    lambda pd: http_download(hf_url(src["repo"], pd[0]), pd[1]),
                    zip(chosen, dests),
                )
            )
        for p in chosen:
            items.append(
                _item(src, "still", Path(p).stem, [f"data:game_stills/{p}"], role)
            )
        log(f"game_stills {role}: {len(chosen)} of {len(names)} images")
    return items


def _tartanair_env(src, data_root, env, role, frames_per_env, seq_len, difficulty):
    """One environment: read a contiguous run of frames of the first trajectory from the remote zips (range
    requests only) and store RGB PNG, depth (fp16 npy), forward flow (fp16 npy) and validity (PNG, 255 = valid).
    TartanAir mask values: 0 = valid correspondence, non-zero = occluded or out of view.
    """
    base = f"{env}/{difficulty}"
    zips = {
        k: remote_zip(hf_url(src["repo"], f"{base}/{k}.zip"))
        for k in ("image_left", "depth_left", "flow_flow", "flow_mask")
    }
    trajs = sorted(
        {n.split("/")[2] for n in zips["image_left"].namelist() if n.count("/") >= 4}
    )
    traj = trajs[0]
    names = {
        k: {
            Path(n).name: n
            for n in z.namelist()
            if f"/{traj}/" in n and not n.endswith("/")
        }
        for k, z in zips.items()
    }
    frame_ids = sorted(
        int(n[:6]) for n in names["image_left"] if n.endswith("_left.png")
    )
    start = (
        frame_ids[len(frame_ids) // 4]
        if len(frame_ids) > 2 * frames_per_env
        else frame_ids[0]
    )
    run = [i for i in frame_ids if start <= i < start + frames_per_env]
    out_dir = Path(data_root) / "tartanair" / env / traj
    for sub in ("image", "depth", "flow"):
        (out_dir / sub).mkdir(parents=True, exist_ok=True)
    have = []
    for i in run:
        fid, nid = f"{i:06d}", f"{i + 1:06d}"
        f_key, m_key = f"{fid}_{nid}_flow.npy", f"{fid}_{nid}_mask.npy"
        if f_key not in names["flow_flow"] or m_key not in names["flow_mask"]:
            break  # keep the run contiguous
        if not (out_dir / "flow" / f"{fid}_valid.png").exists():
            (out_dir / "image" / f"{fid}.png").write_bytes(
                zips["image_left"].read(names["image_left"][f"{fid}_left.png"])
            )
            d = np.load(
                io.BytesIO(
                    zips["depth_left"].read(
                        names["depth_left"][f"{fid}_left_depth.npy"]
                    )
                )
            )
            np.save(
                out_dir / "depth" / f"{fid}.npy", np.minimum(d, 1e4).astype(np.float16)
            )
            fl = np.load(io.BytesIO(zips["flow_flow"].read(names["flow_flow"][f_key])))
            np.save(out_dir / "flow" / f"{fid}.npy", fl.astype(np.float16))
            m = np.load(io.BytesIO(zips["flow_mask"].read(names["flow_mask"][m_key])))
            Image.fromarray(np.where(m == 0, 255, 0).astype(np.uint8)).save(
                out_dir / "flow" / f"{fid}_valid.png"
            )
        have.append(i)
    items, rel = [], f"tartanair/{env}/{traj}"
    for k in range(0, len(have) - seq_len + 1, seq_len):
        seg = have[k : k + seq_len]
        items.append(
            _item(
                src,
                "seq",
                env,
                [f"data:{rel}/image/{i:06d}.png" for i in seg],
                role,
                mv="exact_forward",
                flows=[f"data:{rel}/flow/{i:06d}.npy" for i in seg],
                valid=[f"data:{rel}/flow/{i:06d}_valid.png" for i in seg],
                valid_kind="valid",
                depth=[f"data:{rel}/depth/{i:06d}.npy" for i in seg],
            )
        )
    return items, len(have), traj


def materialise_tartanair(
    src,
    data_root,
    frames_per_env,
    envs_train,
    envs_test,
    seq_len,
    difficulty="Easy",
    workers=6,
    log=print,
):
    """Environments are fetched in parallel threads (one zip set per thread; zip readers are not shared)."""
    jobs = [
        (env, "test" if env in envs_test else "train") for env in envs_train + envs_test
    ]

    def run(job, attempts=3):
        env, role = job
        for attempt in range(attempts):  # frames already written are skipped on a retry
            try:
                return (
                    env,
                    role,
                    _tartanair_env(
                        src, data_root, env, role, frames_per_env, seq_len, difficulty
                    ),
                    None,
                )
            except Exception as exc:
                if attempt == attempts - 1:
                    return env, role, None, exc
                time.sleep(10 * (attempt + 1))

    items = []
    with ThreadPoolExecutor(workers) as pool:
        for env, role, result, exc in pool.map(run, jobs):
            if exc is not None:
                log(f"tartanair {env}: skipped ({exc})")
                continue
            env_items, n, traj = result
            items.extend(env_items)
            log(
                f"tartanair {env} ({role}): {n} frames from {traj}, {len(env_items)} sequences"
            )
    return items


def materialise_gameir(src, data_root, max_clips_train, max_clips_test, log=print):
    """Stream GameIR-SR tars (one sequential HTTP stream per tar, stops early) keeping native 720p and 1440p RGB
    and depth. Frames of a clip are sampled every 10 rendered frames; the leakage group is the town. A dropped
    connection keeps the clips already written (incomplete clips are skipped below).
    """
    items = []
    for role, tars, cap in (
        ("train", src["train_tars"], max_clips_train),
        ("test", src["test_tars"], max_clips_test),
    ):
        clips = {}
        for tar_path in tars:
            try:
                with http_stream(hf_url(src["repo"], tar_path)) as resp:
                    with tarfile.open(fileobj=resp.raw, mode="r|") as tf:
                        for m in tf:
                            parts = m.name.split("/")
                            if (
                                len(parts) < 5
                                or not m.isfile()
                                or not m.name.endswith((".rgb.png", ".depth.png"))
                            ):
                                continue
                            town, clip, res = parts[1], parts[2], parts[3]
                            key = (town, clip)
                            if key not in clips and len(clips) >= cap:
                                break
                            clips.setdefault(key, set()).add(res)
                            dst = (
                                Path(data_root)
                                / "gameir"
                                / town
                                / clip
                                / res
                                / parts[4]
                            )
                            if not dst.exists():
                                dst.parent.mkdir(parents=True, exist_ok=True)
                                tmp = dst.with_suffix(".part")
                                tmp.write_bytes(tf.extractfile(m).read())
                                tmp.replace(dst)
            except Exception as exc:
                log(
                    f"gameir {tar_path}: stream ended early ({exc}); keeping complete clips"
                )
        for (town, clip), res in sorted(clips.items()):
            folder = Path(data_root) / "gameir" / town / clip
            hr = sorted((folder / "1440p").glob("*.rgb.png"), key=natural_key)
            lr = sorted((folder / "720p").glob("*.rgb.png"), key=natural_key)
            if len(hr) < 3 or len(lr) != len(hr):
                continue
            rel = f"gameir/{town}/{clip}"
            items.append(
                _item(
                    src,
                    "seq",
                    town.split("_")[-1],
                    [f"data:{rel}/1440p/{p.name}" for p in hr],
                    role,
                    mv="none",
                    native_lr=[f"data:{rel}/720p/{p.name}" for p in lr],
                    depth=[
                        f"data:{rel}/1440p/{p.name.replace('.rgb.png', '.depth.png')}"
                        for p in hr
                    ],
                    frame_stride=10,
                )
            )
        log(f"gameir {role}: {sum(1 for i in items if i['role'] == role)} clips")
    return items


def materialise_vimeo1080p(src, data_root, n_train, n_val, log=print):
    """Download parquet shards, write each embedded MP4 to disk and register it as a sequence of frames."""
    import pyarrow.parquet as pq

    items = []
    for role, shards, cap in (
        ("train", src["train_shards"], n_train),
        ("val_pinned", src["val_shards"], n_val),
    ):
        written = 0
        for shard in shards:
            if written >= cap:
                break
            local = http_download(
                hf_url(src["repo"], shard), Path(data_root) / "_downloads" / shard
            )
            table = pq.ParquetFile(local)
            for batch in table.iter_batches(
                batch_size=16, columns=["video_file", "video"]
            ):
                for name, blob in zip(
                    batch.column(0).to_pylist(), batch.column(1).to_pylist()
                ):
                    if written >= cap:
                        break
                    stem = Path(name).stem
                    rel = f"vimeo1080p/{role}/{stem}.mp4"
                    dst = Path(data_root) / rel
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    dst.write_bytes(blob)
                    n = video_frame_count(dst)
                    if n < 8:
                        dst.unlink()
                        continue
                    items.append(
                        _item(
                            src,
                            "seq",
                            stem,
                            [f"mp4:{rel}#{i}" for i in range(n)],
                            "test" if role == "val_pinned" else "train",
                            mv="none",
                        )
                    )
                    written += 1
            local.unlink()  # the shard is only a container
        log(f"vimeo1080p {role}: {written} videos")
    return items


# ---------------------------------------------------------------------------------------------------------------
# Item statistics for filtering (run in worker processes)
# ---------------------------------------------------------------------------------------------------------------
def keyframes(item):
    """Frames used to represent an item in deduplication: the still itself, the middle frame of a short clip,
    or frames at 10, 50 and 90 percent of a long sequence (scene content can change within it; the very first and
    last frames are often fades or black).
    """
    frames, n = item["frames"], item["n_frames"]
    if n == 1:
        return [frames[0]]
    if n >= 24:
        return [frames[n // 10], frames[n // 2], frames[n - 1 - n // 10]]
    return [frames[n // 2]]


def thumb_and_stats(task):
    """Worker: decode one frame, return a 256x256 thumbnail and quality statistics."""
    data_root, uri = task
    resolver = Resolver(data_root)
    try:
        img = load_rgb(resolver, uri)
    except Exception as exc:
        return uri, None, {"error": str(exc)[:200]}
    h, w = img.shape[:2]
    grey = img.mean(axis=2)
    rows = grey.mean(axis=1)
    dark = rows < 8
    letterbox = (
        (np.argmax(~dark) + np.argmax(~dark[::-1])) / h if (~dark).any() else 1.0
    )
    small = np.asarray(Image.fromarray(img).resize((256, 256), Image.BILINEAR))
    g = small.mean(axis=2).astype(np.float32)
    lap = g[1:-1, 1:-1] * 4 - g[:-2, 1:-1] - g[2:, 1:-1] - g[1:-1, :-2] - g[1:-1, 2:]
    rg, yb = (
        small[..., 0].astype(np.float32) - small[..., 1],
        0.5 * (small[..., 0].astype(np.float32) + small[..., 1]) - small[..., 2],
    )
    stats = {
        "width": w,
        "height": h,
        "mean": float(grey.mean()),
        "std": float(grey.std()),
        "sharpness": float(lap.var()),
        "letterbox": float(letterbox),
        "colourfulness": float(
            np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean())
        ),
    }
    return uri, small, stats


# ---------------------------------------------------------------------------------------------------------------
# Window extraction for training caches (run in worker processes)
# ---------------------------------------------------------------------------------------------------------------
def _resize(a, size, mode):
    """Resize an array to size = (W, H). Modes: 'image' (uint8 HxWx3, Lanczos), 'nearest' (uint8 HxW),
    'depth' (float HxW, bilinear), 'flow' (float HxWx2, bilinear, vectors rescaled to the new pixel units).
    """
    h, w = a.shape[:2]
    W, H = size
    if (W, H) == (w, h):
        return a
    if mode == "image":
        return np.asarray(Image.fromarray(a).resize((W, H), Image.LANCZOS))
    if mode == "nearest":
        return np.asarray(Image.fromarray(a).resize((W, H), Image.NEAREST))
    if mode == "depth":
        return np.asarray(
            Image.fromarray(a.astype(np.float32), mode="F").resize(
                (W, H), Image.BILINEAR
            )
        )
    if mode == "flow":
        chans = [
            np.asarray(
                Image.fromarray(a[..., c].astype(np.float32), mode="F").resize(
                    (W, H), Image.BILINEAR
                )
            )
            for c in range(2)
        ]
        return np.stack(chans, -1) * np.array([W / w, H / h], np.float32)
    raise ValueError(mode)


def extract_window(task):
    """Worker: decode a window of T frames (plus exact motion, validity and depth when the source has them),
    down-scale by a random factor, take one P x P crop at the same place in all frames, and return motion,
    validity and depth at half resolution.

    Convention: mv[k] is, for every pixel of network frame k, the displacement to its position in network frame
    k-1, in full-resolution (P-grid) pixels (a game engine's motion vector). Sources with exact forward flow
    (TartanAir, Sintel) are played backwards so that their forward flow becomes exactly this vector.
    """
    data_root, item, start, T, P, scale_range, seed = task
    rng = np.random.default_rng(seed)
    resolver = Resolver(data_root)
    exact = item.get("mv") == "exact_forward"
    try:
        idx = list(range(start, start + T))
        frames = load_rgb_many(resolver, [item["frames"][i] for i in idx])
        h, w = frames[0].shape[:2]
        if any(f.shape[:2] != (h, w) for f in frames):
            return seed, None
        flows = [load_flow(resolver, item["flows"][i]) for i in idx] if exact else None
        valids = (
            [
                load_valid(resolver, item["valid"][i], item.get("valid_kind", "valid"))
                for i in idx
            ]
            if exact
            else None
        )
        depths = (
            [load_depth(resolver, item["depth"][i]) for i in idx]
            if item.get("depth")
            else None
        )
    except Exception:
        return seed, None
    r = min(1.0, max(float(rng.uniform(*scale_range)), P / min(h, w) * 1.02))
    W, H = int(round(w * r)), int(round(h * r))
    if min(W, H) < P:
        return seed, None
    frames = [_resize(f, (W, H), "image") for f in frames]
    y, x = int(rng.integers(0, H - P + 1)), int(rng.integers(0, W - P + 1))
    crop = (slice(y, y + P), slice(x, x + P))
    half = (P // 2, P // 2)
    out = {
        "rgb": np.stack([f[crop] for f in frames]),
        "mv": np.zeros((T, P // 2, P // 2, 2), np.float16),
        "valid": np.zeros((T, P // 2, P // 2), np.uint8),
        "depth": np.zeros((T, P // 2, P // 2), np.float16),
        "has_depth": depths is not None,
        "mv_quality": 0.0,
    }
    if depths is not None:
        depths = [_resize(d, (W, H), "depth") for d in depths]
        out["depth"] = np.stack(
            [_resize(normalise_depth(d[crop]), half, "depth") for d in depths]
        ).astype(np.float16)
    if exact:
        flows = [_resize(f, (W, H), "flow")[crop] for f in flows]  # P-grid pixel units
        valids = [_resize(v, (W, H), "nearest")[crop] for v in valids]
        # resampling to half resolution must keep P-grid units, so rescale the half-res vectors back by 2
        mv = np.stack([_resize(f, half, "flow") * 2.0 for f in flows])
        va = np.stack([_resize(v, half, "nearest") for v in valids])
        # reverse time: network frame k holds source frame T-1-k; its previous network frame (k-1) is source
        # frame T-k, which is exactly where the forward flow of source frame T-1-k points
        out["rgb"], out["depth"] = out["rgb"][::-1].copy(), out["depth"][::-1].copy()
        mv, va = mv[::-1].copy(), va[::-1].copy()
        mv[0], va[0] = 0.0, 0  # network frame 0 has no previous frame in the window
        out["mv"], out["valid"], out["mv_quality"] = mv.astype(np.float16), va, 1.0
    return seed, out


def plan_windows(items, T, n, seed, stride=None):
    """Sample n (item, start) windows, spread evenly across items (round-robin over shuffled items)."""
    rng = random.Random(seed)
    usable = [it for it in items if it["n_frames"] >= T]
    rng.shuffle(usable)
    plans = []
    while usable and len(plans) < n:
        for it in usable:
            starts = range(0, it["n_frames"] - T + 1, stride or max(1, T // 2))
            plans.append((it, rng.choice(list(starts))))
            if len(plans) >= n:
                break
    return plans


def still_crop(task):
    """Worker: decode one still or frame and return k random S x S crops after a random down-scale."""
    data_root, uri, k, S, scale_range, seed, min_std = task
    rng = np.random.default_rng(seed)
    try:
        img = load_rgb(Resolver(data_root), uri)
    except Exception:
        return seed, None
    h, w = img.shape[:2]
    r = max(float(rng.uniform(*scale_range)), S / min(h, w) * 1.02)
    if r < 1.0:
        img = _resize(img, (int(round(w * r)), int(round(h * r))), "image")
    h, w = img.shape[:2]
    if min(h, w) < S:
        return seed, None
    crops = []
    for _ in range(k * 3):
        y, x = int(rng.integers(0, h - S + 1)), int(rng.integers(0, w - S + 1))
        c = img[y : y + S, x : x + S]
        if c.mean(axis=2).std() >= min_std:
            crops.append(c)
            if len(crops) == k:
                break
    return seed, np.stack(crops) if crops else None


# ---------------------------------------------------------------------------------------------------------------
# Manifest I/O and discovery of an attached data-pipeline output
# ---------------------------------------------------------------------------------------------------------------
LIST_COLUMNS = ["frames", "flows", "valid", "depth", "native_lr"]


def save_manifest(items, path):
    import pandas as pd

    df = pd.DataFrame(items)
    for c in LIST_COLUMNS:
        if c in df:
            df[c] = df[c].map(lambda v: json.dumps(v) if isinstance(v, list) else None)
    df.to_parquet(path, index=False)
    return df


def load_manifest(path):
    import pandas as pd

    df = pd.read_parquet(path)
    for c in LIST_COLUMNS:
        if c in df:
            df[c] = df[c].map(lambda v: json.loads(v) if isinstance(v, str) else None)
    return df


def items_from_frame(df):
    return [
        {k: v for k, v in row.items() if not (isinstance(v, float) and np.isnan(v))}
        for row in df.to_dict("records")
    ]


def locate_data_root(
    input_root="/kaggle/input", name="nss_manifest.parquet", max_depth=5
):
    """Folder holding the manifest produced by the data-pipeline notebook, if that output is attached."""
    input_root = Path(input_root)
    if not input_root.exists():
        return None
    frontier = [(input_root, 0)]
    while frontier:
        folder, depth = frontier.pop()
        if (folder / name).exists():
            return folder
        if depth < max_depth:
            try:
                frontier.extend((p, depth + 1) for p in folder.iterdir() if p.is_dir())
            except OSError:
                pass
    return None

**`nss_dedup.py`**: pHash / dHash, DINOv2 embeddings on both GPUs, exhaustive GPU neighbour search, SIFT + RANSAC geometric verification in a process pool, synthetic copies for calibration, the duplicate rule and the leakage-safe split.

In [ ]:
%%writefile /kaggle/working/code/nss_dedup.py
"""Perceptual near-duplicate (copy) detection across the combined dataset and leakage-safe splitting.

Two-stage copy detection, because global similarity alone cannot tell "the same picture" from "a similar picture"
(two photos of cherry trees, two skylines or two frames of one game all have high DINOv2 cosine similarity):

1. Candidates: for every keyframe, its nearest neighbours in other leakage groups by DINOv2 cosine similarity
   (exhaustive GPU search) above a high-recall threshold, plus pHash pairs within a loose Hamming radius.
2. Verification: SIFT keypoints matched with Lowe's ratio test and a RANSAC affine fit, also against the mirrored
   image. The score is the number of 8 x 8 grid cells covered by geometrically consistent matches, so a shared logo,
   HUD or caption cannot verify two different frames on its own.

A pair is a duplicate edge if it is verified, or if its cosine similarity or pHash distance lies beyond anything
seen among hard negatives. Thresholds come from synthetic copies (positives) and from negatives that include hard
ones: nearest neighbours between items known to be distinct (curated stills, different rendered environments).

Splitting: leakage groups (same sequence, video, scene family, environment, town) are the unit of assignment. Items
that share a group with a test item, or reach a test item through duplicate edges, are dropped; the rest form
components (groups joined by duplicate edges) that go to validation or training by a deterministic hash.
"""

import hashlib
import io
from collections import defaultdict, deque
from multiprocessing import Pool

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFilter


# ---------------------------------------------------------------------------------------------------------------
# Hashes
# ---------------------------------------------------------------------------------------------------------------
def _dct_matrix(n=32):
    k = np.arange(n)[:, None]
    i = np.arange(n)[None, :]
    m = np.cos(np.pi * (2 * i + 1) * k / (2 * n)) * np.sqrt(2.0 / n)
    m[0] /= np.sqrt(2.0)
    return m.astype(np.float32)


_DCT32 = _dct_matrix(32)


def phash_bits(thumbs):
    """thumbs: uint8 (N, h, w, 3) -> bool (N, 64). DCT of a 32x32 grey image, 8x8 low frequencies (DC excluded
    from the median), each bit = coefficient above the median."""
    grey = np.stack(
        [
            np.asarray(
                Image.fromarray(t).convert("L").resize((32, 32), Image.BILINEAR),
                np.float32,
            )
            for t in thumbs
        ]
    )
    coeffs = _DCT32 @ grey @ _DCT32.T
    low = coeffs[:, :8, :8].reshape(len(thumbs), 64)
    med = np.median(low[:, 1:], axis=1, keepdims=True)
    return low > med


def dhash_bits(thumbs):
    """Horizontal gradient-sign hash on a 9x8 grey image -> bool (N, 64)."""
    grey = np.stack(
        [
            np.asarray(
                Image.fromarray(t).convert("L").resize((9, 8), Image.BILINEAR),
                np.float32,
            )
            for t in thumbs
        ]
    )
    return (grey[:, :, 1:] > grey[:, :, :-1]).reshape(len(thumbs), 64)


def bits_to_hex(bits):
    return [
        "".join(
            f"{int(''.join('1' if b else '0' for b in row[i:i + 4]), 2):x}"
            for i in range(0, 64, 4)
        )
        for row in bits
    ]


def hamming(a, b):
    """Row-wise Hamming distance of two bool (N, 64) arrays."""
    return (a != b).sum(1)


# ---------------------------------------------------------------------------------------------------------------
# Embeddings
# ---------------------------------------------------------------------------------------------------------------
def load_embedder(device, log=print):
    """DINOv2-small (Apache 2.0) from the Hugging Face Hub; returns (callable, name) or (None, None) offline."""
    try:
        from transformers import AutoModel

        model = (
            AutoModel.from_pretrained("facebook/dinov2-small").to(device).eval().half()
        )

        @torch.no_grad()
        def embed(x):  # x: float (N, 3, 224, 224) in [0, 1]
            mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(1, 3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(1, 3, 1, 1)
            out = model(pixel_values=((x - mean) / std).half())
            feat = (
                out.pooler_output
                if getattr(out, "pooler_output", None) is not None
                else out.last_hidden_state[:, 0]
            )
            return F.normalize(feat.float(), dim=1)

        return embed, "dinov2-small"
    except Exception as exc:
        log(f"DINOv2 unavailable ({exc}); embeddings skipped, hashes only")
        return None, None


@torch.no_grad()
def embed_thumbs(thumbs, embedders, devices, batch=256):
    """Embed uint8 thumbnails (N, 256, 256, 3), splitting batches across devices (threads, one per GPU)."""
    from concurrent.futures import ThreadPoolExecutor

    chunks = [(i, thumbs[i : i + batch]) for i in range(0, len(thumbs), batch)]
    out = [None] * len(chunks)

    def work(rank):
        for k in range(rank, len(chunks), len(devices)):
            _, t = chunks[k]
            x = (
                torch.from_numpy(np.ascontiguousarray(t))
                .to(devices[rank])
                .permute(0, 3, 1, 2)
                .float()
                / 255.0
            )
            x = F.interpolate(
                x, size=(224, 224), mode="bilinear", antialias=True, align_corners=False
            )
            out[k] = embedders[rank](x).cpu()

    with ThreadPoolExecutor(len(devices)) as pool:
        list(pool.map(work, range(len(devices))))
    return torch.cat(out).numpy()


# ---------------------------------------------------------------------------------------------------------------
# Exhaustive neighbour search on GPU (cosine for embeddings, Hamming for hashes via +-1 dot products)
# ---------------------------------------------------------------------------------------------------------------
@torch.no_grad()
def pairs_above(feats, threshold, device, chunk=4096):
    """All pairs (i < j) with dot(feats_i, feats_j) >= threshold. Returns int64 (M, 2) and float32 scores."""
    x = torch.from_numpy(np.ascontiguousarray(feats)).to(device).half()
    n = x.shape[0]
    ii, jj, ss = [], [], []
    for a in range(0, n, chunk):
        sim = (x[a : a + chunk] @ x.T).float()
        rows = torch.arange(a, min(a + chunk, n), device=device)[:, None]
        cols = torch.arange(n, device=device)[None, :]
        mask = (sim >= threshold) & (cols > rows)
        r, c = mask.nonzero(as_tuple=True)
        ii.append((r + a).cpu())
        jj.append(c.cpu())
        ss.append(sim[r, c].cpu())
    if not ii:
        return np.zeros((0, 2), np.int64), np.zeros(0, np.float32)
    return torch.stack([torch.cat(ii), torch.cat(jj)], 1).numpy(), torch.cat(ss).numpy()


@torch.no_grad()
def topk_neighbours(query, ref, k, device, query_groups, ref_groups, chunk=2048):
    """For each query row, the k ref rows with the largest dot product, never from the query's own group (which
    also excludes the row itself when query is ref). Returns idx int64 (Q, k) with -1 for empty slots, and scores.
    """
    q = torch.from_numpy(np.ascontiguousarray(query)).to(device).half()
    r = torch.from_numpy(np.ascontiguousarray(ref)).to(device).half()
    qg = torch.from_numpy(np.asarray(query_groups, np.int64)).to(device)
    rg = torch.from_numpy(np.asarray(ref_groups, np.int64)).to(device)
    k = min(k, r.shape[0])
    idx_out, sim_out = [], []
    for a in range(0, q.shape[0], chunk):
        sim = (q[a : a + chunk] @ r.T).float()
        sim.masked_fill_(qg[a : a + chunk, None] == rg[None, :], float("-inf"))
        s, i = sim.topk(k, dim=1)
        i[~torch.isfinite(s)] = -1
        idx_out.append(i.cpu())
        sim_out.append(s.cpu())
    if not idx_out:
        return np.zeros((0, k), np.int64), np.zeros((0, k), np.float32)
    return torch.cat(idx_out).numpy(), torch.cat(sim_out).numpy()


def bits_to_pm1(bits):
    return np.where(bits, 1.0, -1.0).astype(np.float32)


def hamming_threshold_to_dot(h, nbits=64):
    """Hamming distance <= h  <=>  dot of +-1 vectors >= nbits - 2h."""
    return float(nbits - 2 * h)


# ---------------------------------------------------------------------------------------------------------------
# Geometric verification with local features (CPU process pool; tables are shared with forked workers)
# ---------------------------------------------------------------------------------------------------------------
_IMAGES = None  # image array read by forked feature workers
_FEATS = None  # feature table read by forked verification workers
GRID = (
    8  # verification score = occupied cells of a GRID x GRID grid over the first image
)


def _sift(img, n_features):
    import cv2

    grey = cv2.cvtColor(np.ascontiguousarray(img), cv2.COLOR_RGB2GRAY)
    kp, desc = cv2.SIFT_create(nfeatures=n_features).detectAndCompute(grey, None)
    if desc is None or len(kp) < 3:
        return np.zeros((0, 2), np.float32), np.zeros((0, 128), np.uint8)
    pts = np.float32([p.pt for p in kp])
    return pts, np.clip(desc, 0, 255).astype(np.uint8)


def _features_task(args):
    import cv2

    cv2.setNumThreads(1)
    i, n_features = args
    img = _IMAGES[i]
    return _sift(img, n_features), _sift(img[:, ::-1], n_features)


def local_features(images, workers, n_features=400, desc="SIFT features"):
    """SIFT keypoints and uint8 descriptors of every image and of its mirror: list of ((pts, desc), (pts, desc))."""
    global _IMAGES
    from tqdm.auto import tqdm

    _IMAGES = images
    try:
        with Pool(workers) as pool:
            return list(
                tqdm(
                    pool.imap(
                        _features_task,
                        [(i, n_features) for i in range(len(images))],
                        chunksize=32,
                    ),
                    total=len(images),
                    desc=desc,
                )
            )
    finally:
        _IMAGES = None


def match_cells(fa, fb, size=256, ratio=0.8, reproj=4.0):
    """Grid cells of image A covered by matches consistent with one orientation-preserving affine map A -> B.
    Degenerate maps (extreme or collapsing scale) score 0."""
    import cv2

    (pa, da), (pb, db) = fa, fb
    if len(da) < 3 or len(db) < 3:
        return 0
    knn = cv2.BFMatcher(cv2.NORM_L2).knnMatch(
        da.astype(np.float32), db.astype(np.float32), k=2
    )
    good = [m[0] for m in knn if len(m) == 2 and m[0].distance < ratio * m[1].distance]
    if len(good) < 6:
        return 0
    src = pa[[g.queryIdx for g in good]]
    dst = pb[[g.trainIdx for g in good]]
    M, mask = cv2.estimateAffine2D(
        src, dst, method=cv2.RANSAC, ransacReprojThreshold=reproj, maxIters=1000
    )
    if M is None or mask is None:
        return 0
    sv = np.linalg.svd(M[:, :2], compute_uv=False)
    if np.linalg.det(M[:, :2]) <= 0 or sv.min() < 0.2 or sv.max() > 5.0:
        return 0
    inl = src[mask.ravel().astype(bool)]
    if len(inl) < 6:
        return 0
    cells = np.clip((inl * GRID / size).astype(int), 0, GRID - 1)
    return int(len({(int(x), int(y)) for x, y in cells}))


def _verify_task(pair):
    import cv2

    cv2.setNumThreads(1)
    i, j = pair
    a, (b, b_flip) = _FEATS[i][0], _FEATS[j]
    return max(match_cells(a, b), match_cells(a, b_flip))


def verify_pairs(feats, pairs, workers, desc="Geometric verification"):
    """Verification score (covered grid cells, 0..64) for each (i, j) pair of rows of the feature table."""
    global _FEATS
    from tqdm.auto import tqdm

    pairs = [(int(i), int(j)) for i, j in pairs]
    if not pairs:
        return np.zeros(0, np.int32)
    _FEATS = feats
    try:
        with Pool(workers) as pool:
            return np.array(
                list(
                    tqdm(
                        pool.imap(_verify_task, pairs, chunksize=64),
                        total=len(pairs),
                        desc=desc,
                    )
                ),
                np.int32,
            )
    finally:
        _FEATS = None


# ---------------------------------------------------------------------------------------------------------------
# Calibration: synthetic copies (positives); negatives are chosen by the pipeline (random and hard pairs)
# ---------------------------------------------------------------------------------------------------------------
def augment_copy(img, rng, strength="heavy"):
    """A near-duplicate of img as it might appear in another dataset. 'light': rescale, recompress, recolour and
    blur only (what a perceptual hash is meant to survive). 'heavy': additionally re-crop (60 to 95 percent of the
    area), mirror, rotate slightly and overlay text. img: PIL RGB."""
    w, h = img.size
    out = img
    if strength == "heavy":
        a = rng.uniform(0.6, 0.95)
        cw, ch = int(w * np.sqrt(a)), int(h * np.sqrt(a))
        x, y = rng.integers(0, w - cw + 1), rng.integers(0, h - ch + 1)
        out = out.crop((x, y, x + cw, y + ch)).resize((256, 256), Image.BILINEAR)
        if rng.random() < 0.5:
            out = out.transpose(Image.FLIP_LEFT_RIGHT)
        if rng.random() < 0.3:
            out = out.rotate(float(rng.uniform(-4, 4)), resample=Image.BILINEAR)
    else:
        s = rng.uniform(0.4, 1.0)
        out = out.resize((max(16, int(w * s)), max(16, int(h * s))), Image.BILINEAR)
        out = out.resize((256, 256), Image.BILINEAR)
    if rng.random() < 0.5:
        arr = np.asarray(out, np.float32) * rng.uniform(0.8, 1.2) + rng.uniform(-20, 20)
        out = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    if rng.random() < 0.3:
        out = out.filter(ImageFilter.GaussianBlur(float(rng.uniform(0.5, 1.5))))
    if strength == "heavy" and rng.random() < 0.3:
        ImageDraw.Draw(out).text(
            (int(rng.integers(5, 120)), int(rng.integers(5, 220))),
            "SAMPLE 1080p",
            fill=(255, 255, 255),
        )
    buf = io.BytesIO()
    out.save(buf, format="JPEG", quality=int(rng.integers(35, 92)))
    return Image.open(io.BytesIO(buf.getvalue())).convert("RGB")


def synthetic_copies(thumbs, candidates, n_pos, seed):
    """Copies of n_pos random thumbnails (drawn from the candidates index array), half light and half heavy.
    Returns (source indices, uint8 copies (n, 256, 256, 3), strength labels)."""
    rng = np.random.default_rng(seed)
    idx = rng.choice(candidates, size=min(n_pos, len(candidates)), replace=False)
    kinds = np.array(["light" if k % 2 else "heavy" for k in range(len(idx))])
    copies = np.stack(
        [
            np.asarray(
                augment_copy(Image.fromarray(thumbs[i]), rng, kind).resize((256, 256))
            )
            for i, kind in zip(idx, kinds)
        ]
    )
    return idx, copies, kinds


def roc_curve(scores, labels, points=400):
    """False-positive and true-positive rate as the threshold sweeps down the scores (higher = more similar)."""
    order = np.argsort(-scores, kind="stable")
    y = labels[order]
    tpr = np.cumsum(y == 1) / max((labels == 1).sum(), 1)
    fpr = np.cumsum(y == 0) / max((labels == 0).sum(), 1)
    step = max(1, len(fpr) // points)
    return {"fpr": fpr[::step].tolist(), "tpr": tpr[::step].tolist()}


def edge_rule(cos, ham, dham, cells, textured, thr):
    """Duplicate decision for arrays of pair scores. Returns an object array: 'verified' (geometric verification),
    'strong_embedding' (cosine beyond every hard negative), 'exact_hash' (pHash and dHash both nearly identical on
    textured frames) or '' (not a duplicate). cos may be NaN when embeddings are unavailable.
    """
    rule = np.full(len(cells), "", dtype=object)
    h = thr["phash"]["h_exact"]
    exact = textured & (ham <= h) & (dham <= 2 * h + 4)
    rule[exact] = "exact_hash"
    if "embedding" in thr:
        rule[np.nan_to_num(cos, nan=-1.0) >= thr["embedding"]["t_strong"]] = (
            "strong_embedding"
        )
    rule[cells >= thr["verify"]["t_cells"]] = "verified"
    return rule


# ---------------------------------------------------------------------------------------------------------------
# Leakage-safe split
# ---------------------------------------------------------------------------------------------------------------
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, a):
        while self.parent[a] != a:
            self.parent[a] = self.parent[self.parent[a]]
            a = self.parent[a]
        return a

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[max(ra, rb)] = min(ra, rb)


def _components(n, members, edges):
    """Union-find over the given member indices joined by shared group and by edges; returns root per index."""
    uf = UnionFind(n)
    first = {}
    for i, g in members:
        if g in first:
            uf.union(i, first[g])
        else:
            first[g] = i
    for i, j in edges:
        uf.union(i, j)
    return uf


def assign_splits(items, dup_edges, val_fraction, seed, active=None):
    """items: dicts with 'group', 'role' ('train' or 'test') and 'item_id'; dup_edges: (i, j) item index pairs;
    active: keep flags (inactive items get split 'filtered' and are ignored).

    1. Provenance: a leakage group that holds a test item is test-only; its other items are dropped.
    2. Content: non-test items connected to a test item through duplicate edges (transitively, but never through
       shared groups, so one copied clip does not take its whole folder with it) are dropped.
    3. The remaining items form components (shared groups joined by duplicate edges); each component goes to
       validation with probability val_fraction (deterministic hash of its root item), else to training.
    Returns (split list, component id list, drop reason list, report)."""
    n = len(items)
    active = [True] * n if active is None else list(active)
    edges = [
        (int(i), int(j))
        for i, j in dup_edges
        if active[int(i)] and active[int(j)] and int(i) != int(j)
    ]
    is_test = [active[i] and it["role"] == "test" for i, it in enumerate(items)]
    test_groups = {it["group"] for i, it in enumerate(items) if is_test[i]}

    adj = defaultdict(list)
    for i, j in edges:
        adj[i].append(j)
        adj[j].append(i)
    reached = [False] * n
    queue = deque(i for i in range(n) if is_test[i])
    for i in queue:
        reached[i] = True
    while queue:
        a = queue.popleft()
        for b in adj[a]:
            if not reached[b]:
                reached[b] = True
                queue.append(b)

    split, reason = [None] * n, [None] * n
    for i, it in enumerate(items):
        if not active[i]:
            split[i] = "filtered"
        elif is_test[i]:
            split[i] = "test"
        elif it["group"] in test_groups:
            split[i], reason[i] = "drop", "test_group"
        elif reached[i]:
            split[i], reason[i] = "drop", "test_duplicate"

    rest = [i for i in range(n) if split[i] is None]
    rest_set = set(rest)
    uf = _components(
        n,
        [(i, items[i]["group"]) for i in rest],
        [(i, j) for i, j in edges if i in rest_set and j in rest_set],
    )
    for i in rest:
        root = uf.find(i)
        digest = hashlib.md5(f"{seed}/{items[root]['item_id']}".encode()).hexdigest()
        split[i] = (
            "val" if (int(digest, 16) % 10_000) / 10_000.0 < val_fraction else "train"
        )

    # component ids over every active item (groups and all duplicate edges), for the manifest and the report
    act = [i for i in range(n) if active[i]]
    uf_all = _components(n, [(i, items[i]["group"]) for i in act], edges)
    comp = [uf_all.find(i) if active[i] else -1 for i in range(n)]
    uf_dup = _components(n, [], edges)
    dup_sizes = np.bincount([uf_dup.find(i) for i in act], minlength=n) if act else [0]
    rest_sizes = np.bincount([uf.find(i) for i in rest], minlength=n) if rest else [0]
    report = {
        "items": n,
        "active": len(act),
        "dup_edges": len(edges),
        "largest_duplicate_cluster": int(np.max(dup_sizes)),
        "trainval_components": int(np.count_nonzero(rest_sizes)),
        "largest_trainval_component": int(np.max(rest_sizes)),
        "dropped_test_group": reason.count("test_group"),
        "dropped_test_duplicate": reason.count("test_duplicate"),
        "split_counts": {
            s: split.count(s) for s in ("train", "val", "test", "drop", "filtered")
        },
    }
    return split, comp, reason, report

**`nss_pipeline.py`**: the data-pipeline steps (discover, materialise, statistics, filters, hashes, calibration, candidate search and verification, split with audit) and the end-to-end fallback used by the training notebooks.

In [ ]:
%%writefile /kaggle/working/code/nss_pipeline.py
"""Unified data pipeline steps (used cell by cell in the data-pipeline notebook, and end to end by
build_manifest() when the supersampling or frame-generation notebook runs without the pipeline output attached).

discover -> materialise Hugging Face subsets -> keyframe thumbnails and quality statistics -> filters ->
perceptual hashes, embeddings and local features -> threshold calibration -> candidate search and geometric
verification -> leakage-safe split with a cross-split audit (repaired until clean).
"""

import shutil
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd

import nss_data as D
import nss_dedup as DD

FILTER_RULES = {
    "min_side": 256,
    "min_std": 8.0,
    "max_letterbox": 0.35,
    "min_sharpness": {"still": 20.0, "seq": 5.0},
}
DEDUP = {
    "k": 16,  # embedding neighbours per keyframe considered as candidates
    "retrieval_recall": 0.98,  # share of synthetic copies the candidate threshold must retrieve
    "sift_features": 400,
    "min_std_hash": 15.0,  # hash matches only count between keyframes with this much structure
    "max_hash_candidates_per_kf": 20,
    "audit_k": 5,  # nearest reference neighbours verified per val / test keyframe
    "audit_rounds": 3,
}
EDGE_COLUMNS = [
    "kf_a",
    "kf_b",
    "item_a",
    "item_b",
    "cosine",
    "hamming",
    "cells",
    "rule",
]


def discover_kaggle(input_root, cap, seed, log=print):
    items, rows = [], []
    kaggle = [s for s in D.SOURCES if s["host"] == "kaggle"]
    with ThreadPoolExecutor(len(kaggle)) as pool:
        results = list(
            pool.map(
                lambda s: (s, *D.catalogue_kaggle(s, input_root, cap, seed)), kaggle
            )
        )
    for src, its, info in results:
        items.extend(its)
        rows.append(
            {
                "source": src["name"],
                "ref": src["ref"],
                **info,
                "items": len(its),
                "test_items": sum(i["role"] == "test" for i in its),
            }
        )
        if not info.get("found"):
            log(f"{src['name']}: not attached ({src['ref']})")
    return items, pd.DataFrame(rows)


def materialise_hf(data_root, caps, seed, log=print):
    """caps: {"game_stills": (n_train, n_test), "tartanair": frames_per_env, "tartanair_seq": len,
    "gameir": (train_clips, test_clips), "vimeo1080p": (n_train, n_val)}; a cap of 0 skips the source.
    """
    items = []
    steps = [
        (
            "game_stills",
            lambda s: D.materialise_game_stills(
                s, data_root, *caps["game_stills"], seed, log=log
            ),
        ),
        (
            "tartanair",
            lambda s: D.materialise_tartanair(
                s,
                data_root,
                caps["tartanair"],
                s["train_envs"],
                s["test_envs"],
                caps["tartanair_seq"],
                log=log,
            ),
        ),
        (
            "gameir",
            lambda s: D.materialise_gameir(s, data_root, *caps["gameir"], log=log),
        ),
        (
            "vimeo1080p",
            lambda s: D.materialise_vimeo1080p(
                s, data_root, *caps["vimeo1080p"], log=log
            ),
        ),
    ]
    for name, fn in steps:
        cap = caps.get(name)
        if not cap or (isinstance(cap, tuple) and not any(cap)):
            continue
        try:
            items.extend(fn(D.source(name)))
        except Exception as exc:
            log(f"{name}: materialisation failed ({exc}); continuing without it")
    return items


def keyframe_stats(items, data_root, workers):
    """Thumbnails (K, 256, 256, 3) and statistics for every keyframe; returns thumbs, table (one row per keyframe)."""
    tasks, owner = [], []
    for i, it in enumerate(items):
        for uri in D.keyframes(it):
            tasks.append((str(data_root), uri))
            owner.append(i)
    thumbs = np.zeros((len(tasks), 256, 256, 3), np.uint8)
    rows = [None] * len(tasks)
    from tqdm.auto import tqdm

    with Pool(workers) as pool:
        for k, (uri, small, stats) in enumerate(
            tqdm(
                pool.imap(D.thumb_and_stats, tasks, chunksize=4),
                total=len(tasks),
                desc="Keyframe thumbnails and statistics",
            )
        ):
            if small is not None:
                thumbs[k] = small
            rows[k] = {"item": owner[k], "uri": uri, "ok": small is not None, **stats}
    return thumbs, pd.DataFrame(rows)


def apply_filters(items, kf, rules=FILTER_RULES):
    """Item-level keep flags and rejection reasons. Stills are judged on their single keyframe; sequences on the
    median of their keyframes, so a fade-in or one dark shot does not reject a whole video.
    """
    reasons = {}
    for i, g in kf.groupby("item"):
        kind = items[i]["kind"]
        r = []
        if not g["ok"].all():
            r.append("decode_error")
        else:
            if g[["width", "height"]].min(axis=1).min() < rules["min_side"]:
                r.append("too_small")
            if g["std"].median() < rules["min_std"]:
                r.append("flat")
            if g["letterbox"].median() > rules["max_letterbox"]:
                r.append("letterbox")
            if (
                g["sharpness"].median()
                < rules["min_sharpness"]["still" if kind == "still" else "seq"]
            ):
                r.append("blurry")
        reasons[i] = r
    keep = [not reasons.get(i, ["no_keyframe"]) for i in range(len(items))]
    return keep, reasons


def hashes(thumbs):
    return DD.phash_bits(thumbs), DD.dhash_bits(thumbs)


def keyframe_groups(kf, items):
    """Integer leakage-group id of every keyframe (pairs inside one group never matter for the split)."""
    return pd.factorize(pd.Series([items[i]["group"] for i in kf["item"]]))[0]


def _pair_scores(a, b, emb_a, emb_b, ph_a, ph_b, dh_a, dh_b, std_a, std_b, min_std):
    cos = (
        (emb_a * emb_b).sum(1)
        if emb_a is not None
        else np.full(len(a), np.nan, np.float32)
    )
    return {
        "cosine": cos,
        "hamming": DD.hamming(ph_a, ph_b),
        "dhash": DD.hamming(dh_a, dh_b),
        "textured": (std_a >= min_std) & (std_b >= min_std),
    }


def calibrate(
    thumbs,
    kf,
    items,
    emb,
    ph,
    dh,
    feats,
    embedders,
    devices,
    n_pos,
    seed,
    workers,
    cfg=DEDUP,
    log=print,
):
    """Thresholds from synthetic copies (positives, half light and half heavy) against random negatives and hard
    negatives: the nearest keyframe in another group among sources whose groups are known to be distinct.

    * embedding: t_ret retrieves cfg["retrieval_recall"] of the copies (candidate stage, precision comes from
      verification); t_strong lies above the most similar negative (accepted without verification).
    * verify: t_cells lies above the most similar negative's verification score (at least 10 of 64 cells).
    * phash: h_exact lies 4 bits below the closest negative (at most 6); h_cand covers 95 percent of light copies.
    "Most similar negative" is the extreme hard negative or the 99.9th percentile of random negatives.
    Returns (thresholds, per-pair score table)."""
    rng = np.random.default_rng(seed + 1)
    active = kf["active"].values & kf["ok"].values
    std = kf["std"].fillna(0).values
    gid = keyframe_groups(kf, items)
    act = np.nonzero(active)[0]

    src, copies, kinds = DD.synthetic_copies(thumbs, act, n_pos, seed)
    copy_feats = DD.local_features(
        copies, workers, cfg["sift_features"], desc="SIFT features (synthetic copies)"
    )
    emb_c = DD.embed_thumbs(copies, embedders, devices) if embedders else None
    ph_c, dh_c = hashes(copies)
    pos = _pair_scores(
        src,
        src,
        None if emb is None else emb[src],
        emb_c,
        ph[src],
        ph_c,
        dh[src],
        dh_c,
        std[src],
        std[src],
        cfg["min_std_hash"],
    )
    pos["cells"] = DD.verify_pairs(
        feats + copy_feats,
        [(i, len(feats) + m) for m, i in enumerate(src)],
        workers,
        desc="Verification (synthetic copies)",
    )

    i = rng.choice(act, size=2 * len(src))
    j = rng.choice(act, size=2 * len(src))
    keep = gid[i] != gid[j]
    neg_pairs = [np.stack([i[keep], j[keep]], 1)]
    neg_kind = ["random"] * int(keep.sum())
    distinct = {s["name"] for s in D.SOURCES if s.get("distinct_groups")}
    pool = np.array(
        [k for k in act if items[kf["item"].values[k]]["source"] in distinct]
    )
    if len(pool) > 1:
        feats_pool = emb[pool] if emb is not None else DD.bits_to_pm1(ph[pool])
        idx, _ = DD.topk_neighbours(
            feats_pool, feats_pool, 1, devices[0], gid[pool], gid[pool]
        )
        ok = idx[:, 0] >= 0
        hard = np.sort(np.stack([pool[ok], pool[idx[ok, 0]]], 1), axis=1)
        hard = np.unique(hard, axis=0)
        neg_pairs.append(hard)
        neg_kind += ["hard"] * len(hard)
    neg_pairs = np.concatenate(neg_pairs)
    a, b = neg_pairs[:, 0], neg_pairs[:, 1]
    neg = _pair_scores(
        a,
        b,
        None if emb is None else emb[a],
        None if emb is None else emb[b],
        ph[a],
        ph[b],
        dh[a],
        dh[b],
        std[a],
        std[b],
        cfg["min_std_hash"],
    )
    neg["cells"] = DD.verify_pairs(
        feats, neg_pairs, workers, desc="Verification (negatives)"
    )
    log(
        f"calibration: {len(src)} synthetic copies, {neg_kind.count('random')} random and "
        f"{neg_kind.count('hard')} hard negatives"
    )

    hard_mask = np.array(neg_kind) == "hard"

    def worst(values, high=True):
        """Most duplicate-like negative score: the extreme hard negative (trusted labels) or the 99.9th
        percentile of random negatives (a rare random pair may be a genuine duplicate).
        """
        q = 0.999 if high else 0.001
        rand = float(np.quantile(values[~hard_mask], q)) if (~hard_mask).any() else None
        hard = (
            (values[hard_mask].max() if high else values[hard_mask].min())
            if hard_mask.any()
            else None
        )
        vals = [v for v in (rand, hard) if v is not None]
        return float(max(vals) if high else min(vals))

    thr = {}
    if emb is not None:
        cos_p, cos_n = pos["cosine"], neg["cosine"]
        t_ret = float(
            np.clip(np.quantile(cos_p, 1 - cfg["retrieval_recall"]), 0.5, 0.9)
        )
        t_strong = float(np.clip(worst(cos_n) + 0.03, 0.92, 0.995))
        thr["embedding"] = {
            "t_ret": t_ret,
            "t_strong": t_strong,
            "recall_at_t_ret": float((cos_p >= t_ret).mean()),
            "recall_at_t_strong": float((cos_p >= t_strong).mean()),
            "worst_negative": worst(cos_n),
        }
    t_cells = int(np.clip(worst(neg["cells"]) + 2, 10, 40))
    thr["verify"] = {
        "t_cells": t_cells,
        "recall": float((pos["cells"] >= t_cells).mean()),
        "worst_negative": worst(neg["cells"]),
    }
    h_exact = int(np.clip(worst(neg["hamming"], high=False) - 4, 0, 6))
    light = kinds == "light"
    h_cand = int(np.clip(np.quantile(pos["hamming"][light], 0.95), h_exact, 12))
    thr["phash"] = {
        "h_exact": h_exact,
        "h_cand": h_cand,
        "recall_light_at_h_exact": float((pos["hamming"][light] <= h_exact).mean()),
        "recall_light_at_h_cand": float((pos["hamming"][light] <= h_cand).mean()),
        "worst_negative": worst(neg["hamming"], high=False),
    }

    def table(scores, label, kind):
        return pd.DataFrame(
            {
                "label": label,
                "kind": kind,
                "cosine": scores["cosine"],
                "hamming": scores["hamming"],
                "dhash": scores["dhash"],
                "textured": scores["textured"],
                "cells": scores["cells"],
            }
        )

    scores = pd.concat(
        [table(pos, 1, kinds), table(neg, 0, np.array(neg_kind))], ignore_index=True
    )
    scores["rule"] = DD.edge_rule(
        scores["cosine"].values,
        scores["hamming"].values,
        scores["dhash"].values,
        scores["cells"].values,
        scores["textured"].values,
        thr,
    )
    hit = scores["rule"] != ""
    tp, fp = int((hit & (scores["label"] == 1)).sum()), int(
        (hit & (scores["label"] == 0)).sum()
    )
    thr["rule"] = {
        "recall": tp / max(int((scores["label"] == 1).sum()), 1),
        "recall_light": float(
            hit[(scores["label"] == 1) & (scores["kind"] == "light")].mean()
        ),
        "recall_heavy": float(
            hit[(scores["label"] == 1) & (scores["kind"] == "heavy")].mean()
        ),
        "false_positives": fp,
        "precision": tp / max(tp + fp, 1),
    }
    if emb is not None:
        thr["embedding"]["roc"] = DD.roc_curve(
            scores["cosine"].values, scores["label"].values
        )
    return thr, scores


def find_duplicates(
    kf, items, ph, dh, emb, feats, thr, device, workers, cfg=DEDUP, log=print
):
    """Candidate pairs (embedding top-k above t_ret, pHash within h_cand on textured keyframes, never inside one
    group or with a filtered item), scored and verified; returns (edge table, statistics). If hash candidates are
    implausibly many, h_cand is tightened (dark or repetitive frames would otherwise flood verification).
    """
    n = len(kf)
    active = kf["active"].values & kf["ok"].values
    std = kf["std"].fillna(0).values
    gid = keyframe_groups(kf, items)
    act = np.nonzero(active)[0]
    stats = {"keyframes_searched": int(len(act))}
    cand = [np.zeros((0, 2), np.int64)]
    if emb is not None and len(act):
        idx, sim = DD.topk_neighbours(
            emb[act], emb[act], cfg["k"], device, gid[act], gid[act]
        )
        rows = np.repeat(np.arange(len(act)), idx.shape[1])
        ok = (idx.ravel() >= 0) & (sim.ravel() >= thr["embedding"]["t_ret"])
        cand.append(np.stack([act[rows[ok]], act[idx.ravel()[ok]]], 1))
        stats["candidates_embedding"] = int(ok.sum())
    textured = active & (std >= cfg["min_std_hash"])
    tex = np.nonzero(textured)[0]
    h, pairs = thr["phash"]["h_cand"], np.zeros((0, 2), np.int64)
    while len(tex) > 1:
        found, _ = DD.pairs_above(
            DD.bits_to_pm1(ph[tex]), DD.hamming_threshold_to_dot(h), device
        )
        pairs = tex[found]
        pairs = pairs[gid[pairs[:, 0]] != gid[pairs[:, 1]]]
        too_many = len(pairs) > cfg["max_hash_candidates_per_kf"] * n
        if not too_many or h <= thr["phash"]["h_exact"]:
            break
        h -= 1
    thr["phash"]["h_cand_final"] = int(h)
    cand.append(pairs)
    stats["candidates_phash"] = int(len(pairs))
    cand = np.unique(np.sort(np.concatenate(cand), axis=1), axis=0)
    stats["candidates"] = int(len(cand))
    log(f"duplicate candidates: {stats}")
    a, b = cand[:, 0], cand[:, 1]
    sc = _pair_scores(
        a,
        b,
        None if emb is None else emb[a],
        None if emb is None else emb[b],
        ph[a],
        ph[b],
        dh[a],
        dh[b],
        std[a],
        std[b],
        cfg["min_std_hash"],
    )
    cells = DD.verify_pairs(feats, cand, workers, desc="Verification (candidates)")
    rule = DD.edge_rule(
        sc["cosine"], sc["hamming"], sc["dhash"], cells, sc["textured"], thr
    )
    item = kf["item"].values
    edges = pd.DataFrame(
        {
            "kf_a": a,
            "kf_b": b,
            "item_a": item[a],
            "item_b": item[b],
            "cosine": sc["cosine"],
            "hamming": sc["hamming"],
            "cells": cells,
            "rule": rule,
        }
    )
    edges = edges[edges["rule"] != ""].reset_index(drop=True)
    stats.update(
        {
            f"edges_{r}": int((edges["rule"] == r).sum())
            for r in ("verified", "strong_embedding", "exact_hash")
        }
    )
    stats["edges"] = int(len(edges))
    stats["verification_score_hist"] = np.bincount(cells, minlength=65).tolist()
    return edges, stats


def split_items(items, edges, val_fraction, seed, active):
    pairs = list(zip(edges["item_a"], edges["item_b"])) if len(edges) else []
    split, comp, reason, report = DD.assign_splits(
        items, pairs, val_fraction, seed, active
    )
    for it, s, c, r in zip(items, split, comp, reason):
        it["split"], it["component"], it["drop_reason"] = s, int(c), r
    return report


def cross_split_audit(kf, items, ph, dh, emb, feats, thr, device, workers, cfg=DEDUP):
    """Every validation keyframe against training, every test keyframe against training and validation: its
    audit_k nearest reference keyframes by cosine and its nearest by pHash are scored and verified with the same
    rule as the search. Returns (per-split arrays for plots, violation edge table)."""
    split = np.array([items[i]["split"] for i in kf["item"]])
    std = kf["std"].fillna(0).values
    gid = keyframe_groups(kf, items)
    item = kf["item"].values
    pm1 = DD.bits_to_pm1(ph)
    out, violations = {}, []
    for s, refs in (("val", ("train",)), ("test", ("train", "val"))):
        q, r = np.nonzero(split == s)[0], np.nonzero(np.isin(split, refs))[0]
        if not len(q) or not len(r):
            continue
        pairs, res = [], {"reference": "+".join(refs)}
        if emb is not None:
            idx, sim = DD.topk_neighbours(
                emb[q], emb[r], cfg["audit_k"], device, gid[q], gid[r]
            )
            res["max_cosine"] = sim[:, 0]
            rows = np.repeat(np.arange(len(q)), idx.shape[1])
            ok = idx.ravel() >= 0
            pairs.append(np.stack([q[rows[ok]], r[idx.ravel()[ok]]], 1))
        hidx, hsim = DD.topk_neighbours(pm1[q], pm1[r], 1, device, gid[q], gid[r])
        res["min_hamming"] = (64 - hsim[:, 0]) / 2
        ok = hidx[:, 0] >= 0
        pairs.append(np.stack([q[ok], r[hidx[ok, 0]]], 1))
        pairs = np.unique(np.concatenate(pairs), axis=0)
        a, b = pairs[:, 0], pairs[:, 1]
        sc = _pair_scores(
            a,
            b,
            None if emb is None else emb[a],
            None if emb is None else emb[b],
            ph[a],
            ph[b],
            dh[a],
            dh[b],
            std[a],
            std[b],
            cfg["min_std_hash"],
        )
        cells = DD.verify_pairs(feats, pairs, workers, desc=f"Audit {s}")
        best = pd.Series(cells).groupby(a).max()
        res["max_cells"] = best.reindex(q).fillna(0).values
        rule = DD.edge_rule(
            sc["cosine"], sc["hamming"], sc["dhash"], cells, sc["textured"], thr
        )
        bad = rule != ""
        violations.append(
            pd.DataFrame(
                {
                    "kf_a": a[bad],
                    "kf_b": b[bad],
                    "item_a": item[a[bad]],
                    "item_b": item[b[bad]],
                    "cosine": sc["cosine"][bad],
                    "hamming": sc["hamming"][bad],
                    "cells": cells[bad],
                    "rule": rule[bad],
                    "split": s,
                }
            )
        )
        out[s] = res
    viol = (
        pd.concat(violations, ignore_index=True)
        if violations
        else pd.DataFrame(columns=EDGE_COLUMNS + ["split"])
    )
    return out, viol


def split_and_audit(
    kf,
    items,
    edges,
    active,
    ph,
    dh,
    emb,
    feats,
    thr,
    device,
    workers,
    val_fraction,
    seed,
    cfg=DEDUP,
    log=print,
):
    """Split, audit, and while the audit finds cross-split duplicates (pairs the top-k candidate search missed),
    add them as edges and split again. Returns (report, audit, remaining violations, final edges).
    """
    for rnd in range(cfg["audit_rounds"]):
        report = split_items(items, edges, val_fraction, seed, active)
        audit, viol = cross_split_audit(
            kf, items, ph, dh, emb, feats, thr, device, workers, cfg
        )
        report["audit_rounds"] = rnd + 1
        report["audit_violations"] = int(len(viol))
        if not len(viol):
            break
        log(
            f"audit round {rnd + 1}: {len(viol)} cross-split duplicates; adding them as edges and re-splitting"
        )
        edges = pd.concat([edges, viol[EDGE_COLUMNS]], ignore_index=True)
        edges = edges.drop_duplicates(["kf_a", "kf_b"]).reset_index(drop=True)
    return report, audit, viol, edges


def load_embedders(devices, log=print):
    """One DINOv2 copy per GPU, or [] if the weights are unavailable."""
    embedders = []
    for dvc in devices:
        e, _ = DD.load_embedder(dvc, log=log)
        if e is None:
            return []
        embedders.append(e)
    return embedders


def build_manifest(data_root, input_root, cfg, devices, log=print):
    """End-to-end fallback used by the training notebooks when no pipeline output is attached."""
    dd = {**DEDUP, **cfg.get("dedup", {})}
    Path(data_root).mkdir(parents=True, exist_ok=True)
    items, _ = discover_kaggle(input_root, cfg["list_cap_files"], cfg["seed"], log=log)
    items += materialise_hf(data_root, cfg["hf_caps"], cfg["seed"], log=log)
    thumbs, kf = keyframe_stats(items, data_root, cfg["workers"])
    keep, _ = apply_filters(items, kf)
    kf["active"] = [keep[i] for i in kf["item"]]
    ph, dh = hashes(thumbs)
    embedders = load_embedders(devices, log=log)
    emb = DD.embed_thumbs(thumbs, embedders, devices) if embedders else None
    feats = DD.local_features(thumbs, cfg["workers"], dd["sift_features"])
    thr, _ = calibrate(
        thumbs,
        kf,
        items,
        emb,
        ph,
        dh,
        feats,
        embedders,
        devices,
        cfg["calibration_pairs"],
        cfg["seed"],
        cfg["workers"],
        dd,
        log=log,
    )
    edges, _ = find_duplicates(
        kf, items, ph, dh, emb, feats, thr, devices[0], cfg["workers"], dd, log=log
    )
    report, _, viol, _ = split_and_audit(
        kf,
        items,
        edges,
        keep,
        ph,
        dh,
        emb,
        feats,
        thr,
        devices[0],
        cfg["workers"],
        cfg["val_fraction"],
        cfg["seed"],
        dd,
        log=log,
    )
    if len(viol):
        log(
            f"warning: {len(viol)} cross-split duplicates remain after {report['audit_rounds']} audit rounds"
        )
    D.save_manifest(items, Path(data_root) / "nss_manifest.parquet")
    log(f"manifest: {report['split_counts']}")
    shutil.rmtree(Path(data_root) / "_downloads", ignore_errors=True)
    return items, report

## 2. Setup

Imports the library, creates the output folders and logger, and reads the optional `HF_TOKEN` Kaggle secret into the environment (never printed or saved). `RUN` keeps per-stage timings for the summary.

In [ ]:
import os
import sys
import time

sys.path.insert(0, "/kaggle/working/code")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import nss_common as C
import nss_data as D
import nss_dedup as DD
import nss_pipeline as PL

RUN = C.Run("nss_data")
log = RUN.log.info
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    log("HF_TOKEN secret loaded")
except Exception:
    log("no HF_TOKEN secret: anonymous Hugging Face access")

## 3. Hardware check

Detects GPUs, CPU cores, RAM and free disk at run time and saves `metrics/hardware.json`. Both GPUs compute embeddings; all CPU cores decode images and run SIFT verification. Expected output: two Tesla T4 with about 15 GB each.

In [ ]:
HW = C.hardware_report()
RUN.save_json(HW, "hardware.json")
print(pd.Series({k: v for k, v in HW.items() if k != "gpu_list"}).to_string())
print(pd.DataFrame(HW["gpu_list"]).to_string(index=False))
assert HW["gpus"] >= 2, "Select the GPU T4 x2 accelerator (Settings > Accelerator)."
DEVICES = [torch.device("cuda", i) for i in range(HW["gpus"])]

## 4. Configuration

All caps and thresholds in one place. Hugging Face caps are sized to keep the materialised data under about 14 GB of the 20 GB `/kaggle/working` quota (the first run used 13.4 GB); the cell lowers them when free disk is short.

* `hf_caps`: game frames (train, test), TartanAir frames per environment and sequence length, GameIR clips (train, test), Vimeo-1080p videos (train, test).
* `calibration_pairs`: synthetic copies used to calibrate the duplicate rule (half light, half heavy edits).
* `dedup`: neighbours per keyframe (`k`), the share of synthetic copies the candidate stage must retrieve, SIFT keypoints per image, minimum grey standard deviation for hash matches, and audit depth and rounds.
* `val_fraction`: share of train / validation components assigned to validation.

In [ ]:
QUICK_RUN = False
CFG = {
    "seed": 7,
    "list_cap_files": 4000 if QUICK_RUN else 40000,
    "hf_caps": {
        "game_stills": (150, 40) if QUICK_RUN else (1200, 250),
        "tartanair": 40 if QUICK_RUN else 120,
        "tartanair_seq": 24,
        "gameir": (4, 4) if QUICK_RUN else (40, 30),
        "vimeo1080p": (8, 4) if QUICK_RUN else (160, 40),
    },
    "calibration_pairs": 300 if QUICK_RUN else 1500,
    "dedup": dict(PL.DEDUP),
    "val_fraction": 0.05,
    "workers": os.cpu_count(),
}
DATA_ROOT = C.WORK / "nss_data"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
free_gb = HW["disk_free_gb_working"]
if (
    free_gb < 16
):  # keep room for the training notebooks' own outputs when this output is reused
    f = max(free_gb - 4, 2) / 12
    gs, ta, gi, vm = (
        CFG["hf_caps"]["game_stills"],
        CFG["hf_caps"]["tartanair"],
        CFG["hf_caps"]["gameir"],
        CFG["hf_caps"]["vimeo1080p"],
    )
    CFG["hf_caps"].update(
        {
            "game_stills": (int(gs[0] * f), int(gs[1] * f)),
            "tartanair": int(ta * f),
            "gameir": (int(gi[0] * f), int(gi[1] * f)),
            "vimeo1080p": (int(vm[0] * f), int(vm[1] * f)),
        }
    )
RUN.save_json({"quick_run": QUICK_RUN, "cfg": CFG, "sources": D.SOURCES}, "config.json")
print(pd.Series(CFG["hf_caps"]).to_string())
print(pd.Series(CFG["dedup"]).to_string())

## 5. Kaggle source discovery

Lists each attached Kaggle dataset in parallel threads. A capped randomised walk lists each directory completely, so a sequence is never cut in half. Degraded copies are dropped by path token. Items are built from the listing: stills, sequences grouped by folder, and Sintel scenes with their flow and occlusion files. When a source ships an official test list (Vimeo triplets), clips are read straight from it instead of hoping a random walk finds them. Test roles: `DIV2K_valid_HR`, Vid4, the Vimeo triplet test list, and the Sintel `market` and `cave` scene families. Expected output: one row per source with files listed, items and test items.

In [ ]:
with RUN.stage("kaggle_discovery"):
    kaggle_items, discovery = PL.discover_kaggle(
        C.INPUT_ROOT, CFG["list_cap_files"], CFG["seed"], log=log
    )
RUN.save_csv(discovery, "discovery_kaggle.csv", index=False)
print(discovery.to_string(index=False))

## 6. Hugging Face materialisation

Downloads only what is needed into `nss_data/`. Each thread keeps one HTTP connection alive, range reads are length-checked and retried, and an expired CDN link is re-resolved:

* **game_stills** (`ericphann/video-game-super-resolution`): 1080p frames from game scenes; `test-hr` pinned to test.
* **tartanair** (`theairlabcmu/tartanair`, BSD-3): contiguous runs from 12 training and 2 held-out environments, read member by member from the remote zips. Each run keeps RGB, depth, forward optical flow and the flow mask (occlusion / out of view). A failed environment is retried twice, resuming from the frames already written.
* **gameir** (`LLLebin/GameIR`, MIT): CARLA / Unreal Engine 4 clips read from the mini tars in one sequential stream, keeping native 1440p and 720p renders and depth; town 05 is the official test town.
* **vimeo1080p** (`danjacobellis/vimeo1080p`): MP4 videos extracted from two training parquet shards and one validation shard (validation pinned to test).

Each source is optional: a failure is logged and the pipeline continues. Expected output: per-source counts and the disk used.

In [ ]:
with RUN.stage("hf_materialisation"):
    hf_items = PL.materialise_hf(DATA_ROOT, CFG["hf_caps"], CFG["seed"], log=log)
used_gb = sum(p.stat().st_size for p in DATA_ROOT.rglob("*") if p.is_file()) / 1e9
log(f"Hugging Face items: {len(hf_items)}, data root size {used_gb:.2f} GB")
ITEMS = kaggle_items + hf_items
cat = pd.DataFrame(
    [
        {
            "source": i["source"],
            "kind": i["kind"],
            "role": i["role"],
            "frames": i["n_frames"],
            "motion": i.get("mv", "none"),
            "depth": bool(i.get("depth")),
        }
        for i in ITEMS
    ]
)
summary = (
    cat.groupby(["source", "kind", "role"])
    .agg(
        items=("frames", "size"),
        frames=("frames", "sum"),
        exact_motion=("motion", lambda m: int((m == "exact_forward").sum())),
        with_depth=("depth", "sum"),
    )
    .reset_index()
)
RUN.save_csv(summary, "catalogue_summary.csv", index=False)
print(summary.to_string(index=False))

## 7. Catalogue overview

Item and frame counts per source and role, and the share of data carrying game-rendering metadata (exact motion vectors, depth). Expected output: two bar charts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
summary.pivot_table(
    index="source", columns="role", values="items", aggfunc="sum"
).fillna(0).plot.barh(ax=axes[0], stacked=True)
axes[0].set_xlabel("items (stills or sequences)")
axes[0].set_title("Items by source and role")
meta = summary.groupby("source")[["frames", "exact_motion", "with_depth"]].sum()
meta.plot.barh(ax=axes[1], logx=True)
axes[1].set_title("Frames, sequences with exact motion, sequences with depth")
RUN.show(fig, "catalogue")

## 8. Keyframe thumbnails and quality statistics

Decodes one keyframe per still or short clip and three per long sequence, in a process pool over all CPU cores. Long sequences use frames at 10, 50 and 90 percent, because first and last frames are often fades or black. Each keyframe keeps a 256x256 thumbnail and statistics: size, grey mean and contrast, Laplacian-variance sharpness, letterbox fraction and colourfulness. Expected output: a progress bar and a statistics table.

In [ ]:
with RUN.stage("keyframe_stats"):
    THUMBS, KF = PL.keyframe_stats(ITEMS, DATA_ROOT, CFG["workers"])
KF["source"] = [ITEMS[i]["source"] for i in KF["item"]]
RUN.save_csv(KF, "keyframe_stats.csv", index=False)
print(
    KF.groupby("source")[
        ["width", "height", "std", "sharpness", "letterbox", "colourfulness"]
    ]
    .median()
    .round(2)
    .to_string()
)
print(f"keyframes: {len(KF)}, decode failures: {int((~KF['ok']).sum())}")

## 9. Filtering unsuitable data

Rejects items whose keyframes fail to decode or are smaller than 256 px, flat (grey standard deviation below 8), heavily letterboxed (more than 35 percent black rows) or blurry (Laplacian variance below 20 for stills, 5 for video). Sequences are judged on the median of their keyframes, so one dark shot or a fade does not reject a whole video. Rejected items stay in the manifest with split `filtered` and are left out of duplicate search, so the decision is auditable. Expected output: rejection counts by reason and source, distributions with thresholds, and examples of rejected keyframes from every source and reason.

In [ ]:
with RUN.stage("filters"):
    KEEP, REASONS = PL.apply_filters(ITEMS, KF)
    KF["active"] = [KEEP[i] for i in KF["item"]]
rej = pd.DataFrame(
    [
        {"source": ITEMS[i]["source"], "reason": r}
        for i, rs in REASONS.items()
        for r in rs
    ],
    columns=["source", "reason"],
)
if len(rej):
    print(rej.value_counts().unstack(fill_value=0).to_string())
print(f"kept {sum(KEEP)} of {len(ITEMS)} items")
rules = PL.FILTER_RULES
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for ax, (col, thrs, logx) in zip(
    axes,
    [
        ("std", [rules["min_std"]], False),
        ("sharpness", list(rules["min_sharpness"].values()), True),
        ("letterbox", [rules["max_letterbox"]], False),
    ],
):
    for src, g in KF.groupby("source"):
        vals = g[col].dropna()
        ax.hist(np.log10(vals + 1e-3) if logx else vals, bins=40, alpha=0.4, label=src)
    for t in thrs:
        ax.axvline(np.log10(t) if logx else t, color="k", ls="--")
    ax.set_title(f"{'log10 ' if logx else ''}{col}")
axes[0].legend(fontsize=6)
RUN.show(fig, "quality_distributions")
rej_kf = pd.DataFrame(
    [
        {"k": k, "source": KF["source"][k], "reason": ",".join(REASONS[i])}
        for k, i in enumerate(KF["item"])
        if not KEEP[i]
    ],
    columns=["k", "source", "reason"],
)
if len(rej_kf):
    show = rej_kf.groupby(["source", "reason"]).head(2).head(16)
    fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axes.flat, show.iterrows()):
        ax.imshow(THUMBS[r["k"]])
        ax.set_title(f"{r['source']}: {r['reason']}", fontsize=7)
    RUN.show(fig, "rejected_examples")

## 10. Perceptual hashes

64-bit pHash (DCT of a 32x32 grey image, 8x8 low frequencies against their median) and dHash (horizontal gradient signs on 9x8) for every keyframe. Hashes catch near-exact copies (rescaled, recompressed, recoloured) but not crops or mirrored copies; embeddings and verification cover those. Expected output: hash bit-balance check.

In [ ]:
with RUN.stage("hashes"):
    PH, DH = PL.hashes(THUMBS)
print(
    f"pHash mean bit = {PH.mean():.3f}, dHash mean bit = {DH.mean():.3f} (about 0.5 means balanced bits)"
)

## 11. Embeddings on both GPUs

DINOv2-small (Apache 2.0) global descriptors (384-d, L2-normalised) for every thumbnail, with batches split across the two GPUs, one thread per GPU. Here they only *retrieve* candidates: global similarity is high for look-alikes as well as copies, so it cannot decide alone. If the weights cannot be downloaded, the pipeline falls back to hash candidates. Expected output: embedding matrix shape and time.

In [ ]:
with RUN.stage("embeddings"):
    EMBEDDERS = PL.load_embedders(DEVICES, log=log)
    t0 = time.time()
    EMB = DD.embed_thumbs(THUMBS, EMBEDDERS, DEVICES) if EMBEDDERS else None
log(f"embeddings: {None if EMB is None else EMB.shape} in {time.time() - t0:.0f} s")

## 12. Local features for geometric verification

SIFT keypoints (up to `sift_features` per image) and descriptors for every thumbnail and for its mirror image, computed in a process pool over all CPU cores and kept in RAM (uint8 descriptors). Verification later matches two keyframes with Lowe's ratio test and fits one affine map with RANSAC. The score is the number of cells of an 8x8 grid covered by consistent matches, so a shared logo, HUD or caption alone cannot verify two different frames. Expected output: a progress bar and keypoint statistics.

In [ ]:
with RUN.stage("local_features"):
    FEATS = DD.local_features(THUMBS, CFG["workers"], CFG["dedup"]["sift_features"])
n_kp = np.array([len(f[0][0]) for f in FEATS])
print(
    f"keypoints per keyframe: median {np.median(n_kp):.0f}, "
    f"under 20: {int((n_kp < 20).sum())} of {len(n_kp)}"
)

## 13. Threshold calibration

Positives are synthetic copies of random keyframes. Half are *light* edits: rescale, JPEG quality 35 to 92, colour and brightness shift, blur. Half are *heavy*: these also crop to 60 to 95 percent of the area, mirror, rotate slightly and overlay text. Negatives are random pairs from different groups plus *hard* negatives: each keyframe's nearest neighbour in another group, among sources whose groups are known to be distinct (DIV2K images, TartanAir environments, Sintel scene families).

Thresholds, from the scores of the most similar negative (the extreme hard negative, or the 99.9th percentile of random negatives):

* candidate cosine `t_ret` retrieves 98 percent of the copies; precision comes from verification;
* strong cosine `t_strong` lies above the most similar negative, so such pairs count without verification;
* verification `t_cells` lies above the most similar negative's score (at least 10 of 64 cells);
* exact hash `h_exact` lies 4 bits below the closest negative's pHash distance (at most 6, also confirmed by dHash on textured frames); `h_cand` covers 95 percent of light copies and only proposes candidates.

Expected output: the thresholds, recall of the combined rule on light and heavy copies, false positives on the negatives, and four diagnostic plots.

In [ ]:
with RUN.stage("calibration"):
    THR, CAL = PL.calibrate(
        THUMBS,
        KF,
        ITEMS,
        EMB,
        PH,
        DH,
        FEATS,
        EMBEDDERS,
        DEVICES,
        CFG["calibration_pairs"],
        CFG["seed"],
        CFG["workers"],
        CFG["dedup"],
        log=log,
    )
RUN.save_json(
    {k: {m: x for m, x in v.items() if m != "roc"} for k, v in THR.items()},
    "dedup_calibration.json",
)
RUN.save_csv(CAL, "calibration_scores.csv", index=False)
for key, val in THR.items():
    print(
        key,
        {
            m: (round(x, 4) if isinstance(x, float) else x)
            for m, x in val.items()
            if m != "roc"
        },
    )
groups = {
    "copy (light)": (CAL["label"] == 1) & (CAL["kind"] == "light"),
    "copy (heavy)": (CAL["label"] == 1) & (CAL["kind"] == "heavy"),
    "hard negative": CAL["kind"] == "hard",
    "random negative": CAL["kind"] == "random",
}
fig, axes = plt.subplots(1, 4, figsize=(20, 3.8))
if EMB is not None:
    for name, m in groups.items():
        axes[0].hist(CAL.loc[m, "cosine"], bins=50, range=(0, 1), alpha=0.5, label=name)
    for t in (THR["embedding"]["t_ret"], THR["embedding"]["t_strong"]):
        axes[0].axvline(t, color="k", ls="--")
    axes[0].set_title("DINOv2 cosine (t_ret, t_strong)")
    axes[0].legend(fontsize=7)
    roc = THR["embedding"]["roc"]
    axes[3].plot(roc["fpr"], roc["tpr"])
    axes[3].set_xscale("symlog", linthresh=1e-3)
    axes[3].set_xlabel("false-positive rate")
    axes[3].set_ylabel("recall")
    axes[3].set_title("ROC of cosine alone")
for name, m in groups.items():
    axes[1].hist(CAL.loc[m, "cells"], bins=33, range=(0, 65), alpha=0.5, label=name)
axes[1].axvline(THR["verify"]["t_cells"] - 0.5, color="k", ls="--")
axes[1].set_yscale("log")
axes[1].set_title("verification: covered grid cells (t_cells)")
for name, m in groups.items():
    axes[2].hist(CAL.loc[m, "hamming"], bins=33, range=(0, 64), alpha=0.5, label=name)
for t in (THR["phash"]["h_exact"], THR["phash"]["h_cand"]):
    axes[2].axvline(t + 0.5, color="k", ls="--")
axes[2].set_title("pHash Hamming (h_exact, h_cand)")
RUN.show(fig, "dedup_calibration")

## 14. Copy detection across the combined dataset

Candidates come from two sources. The first is each kept keyframe's `k` nearest DINOv2 neighbours outside its own leakage group with cosine at least `t_ret`, from an exhaustive half-precision search on the GPU. The second is pHash pairs within `h_cand` between textured keyframes. If the hash candidates are implausibly many, `h_cand` is tightened. Every candidate is verified with SIFT + RANSAC in a process pool. A pair becomes a duplicate edge if it is `verified`, `strong_embedding` or `exact_hash` (section 13). Expected output: candidate and edge counts per rule, a cross-source edge matrix (which datasets overlap) and examples of edges for visual checking, cross-source pairs first.

In [ ]:
with RUN.stage("duplicate_search"):
    EDGES, DUP_STATS = PL.find_duplicates(
        KF,
        ITEMS,
        PH,
        DH,
        EMB,
        FEATS,
        THR,
        DEVICES[0],
        CFG["workers"],
        CFG["dedup"],
        log=log,
    )
RUN.save_csv(EDGES, "duplicate_pairs.csv", index=False)
RUN.save_json(DUP_STATS, "duplicate_search.json")
print(
    pd.Series(
        {k: v for k, v in DUP_STATS.items() if k != "verification_score_hist"}
    ).to_string()
)
if len(EDGES):
    EDGES["source_a"] = [ITEMS[i]["source"] for i in EDGES["item_a"]]
    EDGES["source_b"] = [ITEMS[i]["source"] for i in EDGES["item_b"]]
    print(pd.crosstab(EDGES["source_a"], EDGES["source_b"]).to_string())
    cross = EDGES["source_a"] != EDGES["source_b"]
    show_pairs = (
        pd.concat(
            [
                EDGES[cross].sample(frac=1, random_state=0),
                EDGES[~cross].sample(frac=1, random_state=0),
            ]
        )
        .groupby("rule", group_keys=False)
        .head(4)
        .head(10)
    )
    fig, axes = plt.subplots(
        2, len(show_pairs), figsize=(2.2 * len(show_pairs), 4.8), squeeze=False
    )
    for c, (_, r) in enumerate(show_pairs.iterrows()):
        for row, kf_col in ((0, "kf_a"), (1, "kf_b")):
            axes[row, c].imshow(THUMBS[r[kf_col]])
            axes[row, c].axis("off")
        axes[1, c].set_title(KF["source"][r["kf_b"]], fontsize=7)
        axes[0, c].set_title(
            f"{KF['source'][r['kf_a']]}\n{r['rule']} cos {r['cosine']:.2f} cells {r['cells']}",
            fontsize=7,
        )
    RUN.show(fig, "duplicate_examples")

## 15. Leakage-safe split with audit

1. **Provenance.** A leakage group (sequence, video, Sintel scene family, TartanAir environment, GameIR town) that holds a test item is test-only, and its other items are dropped.
2. **Content.** Non-test items connected to a test item through duplicate edges are dropped. The walk runs through duplicate edges only, never through shared groups, so one copied Vimeo-90K clip does not take its 1,000-clip folder with it.
3. **Train and validation.** The remaining items form components (groups joined by duplicate edges). Each component goes to validation with probability `val_fraction` (deterministic hash) or to training.
4. **Audit.** Every validation keyframe is compared with its `audit_k` nearest training keyframes, and every test keyframe with its nearest training and validation keyframes, by cosine and by pHash, using the same rule. Any duplicate the top-`k` search missed becomes an edge and the split is repeated, up to `audit_rounds` times. The cell fails if duplicates remain.

Expected output: the split report, drop reasons, items per source and split (table and chart), and zero audit violations.

In [ ]:
with RUN.stage("split_and_audit"):
    REPORT, AUDIT, VIOL, EDGES = PL.split_and_audit(
        KF,
        ITEMS,
        EDGES,
        KEEP,
        PH,
        DH,
        EMB,
        FEATS,
        THR,
        DEVICES[0],
        CFG["workers"],
        CFG["val_fraction"],
        CFG["seed"],
        CFG["dedup"],
        log=log,
    )
RUN.save_json(REPORT, "split_report.json")
RUN.save_csv(
    EDGES, "duplicate_pairs.csv", index=False
)  # final edges, including audit repairs
RUN.save_csv(VIOL, "leakage_violations.csv", index=False)
print(pd.Series({k: v for k, v in REPORT.items() if k != "split_counts"}).to_string())
print(pd.Series(REPORT["split_counts"]).to_string())
split_table = pd.crosstab(
    pd.Series([i["source"] for i in ITEMS], name="source"),
    pd.Series([i["split"] for i in ITEMS], name="split"),
).reindex(columns=["train", "val", "test", "drop", "filtered"], fill_value=0)
RUN.save_csv(split_table, "split_by_source.csv")
print(split_table.to_string())
drops = pd.crosstab(
    pd.Series([i["source"] for i in ITEMS if i["drop_reason"]], name="source"),
    pd.Series(
        [i["drop_reason"] for i in ITEMS if i["drop_reason"]], name="drop reason"
    ),
)
print(drops.to_string() if len(drops) else "no items dropped")
fig, ax = plt.subplots(figsize=(10, 4.5))
split_table.plot.barh(ax=ax, stacked=True)
ax.set_xlabel("items")
ax.set_title("Items per source and split")
RUN.show(fig, "split_by_source")
assert not len(
    VIOL
), f"{len(VIOL)} cross-split duplicates remain; see metrics/leakage_violations.csv"

## 16. Audit statistics

How close each validation and test keyframe comes to the data it is compared with: the highest cosine similarity, the highest verification score and the smallest pHash distance among its nearest reference keyframes, each against the duplicate thresholds. Values beyond a threshold are allowed only when the other criteria reject the pair (a high cosine without geometric support is a look-alike, not a copy). Expected output: three histograms and the audit table.

In [ ]:
rows = []
fig, axes = plt.subplots(1, 3, figsize=(18, 3.6))
for s, res in AUDIT.items():
    label = f"{s} vs {res['reference']}"
    if "max_cosine" in res:
        axes[0].hist(res["max_cosine"], bins=60, range=(0, 1), alpha=0.5, label=label)
    axes[1].hist(res["max_cells"], bins=33, range=(0, 65), alpha=0.5, label=label)
    axes[2].hist(res["min_hamming"], bins=33, range=(0, 64), alpha=0.5, label=label)
    for metric, vals, worst in (
        ("max cosine", res.get("max_cosine"), np.max),
        ("max verification cells", res["max_cells"], np.max),
        ("min pHash Hamming", res["min_hamming"], np.min),
    ):
        if vals is not None:
            rows.append(
                {
                    "split": s,
                    "reference": res["reference"],
                    "metric": metric,
                    "worst": float(worst(vals)),
                    "median": float(np.median(vals)),
                }
            )
if "embedding" in THR:
    axes[0].axvline(THR["embedding"]["t_strong"], color="k", ls="--")
axes[1].axvline(THR["verify"]["t_cells"] - 0.5, color="k", ls="--")
axes[1].set_yscale("log")
axes[2].axvline(THR["phash"]["h_exact"] + 0.5, color="k", ls="--")
axes[0].set_title("nearest reference keyframe: cosine (t_strong)")
axes[1].set_title("best verification score (t_cells)")
axes[2].set_title("nearest pHash Hamming (h_exact)")
for ax in axes:
    ax.legend(fontsize=7)
RUN.show(fig, "leakage_audit")
audit = pd.DataFrame(rows)
RUN.save_csv(audit, "leakage_audit.csv", index=False)
print(audit.to_string(index=False))

## 17. Manifest and dataset card

Saves the manifest (`nss_data/nss_manifest.parquet`) and a dataset card. The card lists every source with its host, licence statement, content type, item counts per split, and whether it carries exact motion vectors or depth. Licences are as stated by the hosting page; several research datasets restrict commercial use (see `RESEARCH.md`). Expected output: the card table.

In [ ]:
with RUN.stage("manifest"):
    D.save_manifest(ITEMS, DATA_ROOT / "nss_manifest.parquet")
card = []
for s in D.SOURCES:
    its = [i for i in ITEMS if i["source"] == s["name"]]
    card.append(
        {
            "source": s["name"],
            "host": s["host"],
            "id": s.get("ref") or s.get("repo"),
            "licence": s["licence"],
            "content": s.get("content"),
            "items": len(its),
            **{
                sp: sum(i["split"] == sp for i in its)
                for sp in ("train", "val", "test", "drop", "filtered")
            },
            "exact_motion": any(i.get("mv") == "exact_forward" for i in its),
            "depth": any(bool(i.get("depth")) for i in its),
        }
    )
card = pd.DataFrame(card)
RUN.save_csv(card, "dataset_card.csv", index=False)
print(card.to_string(index=False))

## 18. Summary

Stage times and the headline numbers of the pipeline. Expected output: a summary table and a stage-time bar chart.

In [ ]:
summary = {
    "items": len(ITEMS),
    "keyframes": len(KF),
    "candidates": DUP_STATS["candidates"],
    "duplicate_edges": int(len(EDGES)),
    **{f"edges_{k}": int(v) for k, v in EDGES["rule"].value_counts().items()},
    **REPORT["split_counts"],
    "dropped_test_group": REPORT["dropped_test_group"],
    "dropped_test_duplicate": REPORT["dropped_test_duplicate"],
    "largest_duplicate_cluster": REPORT["largest_duplicate_cluster"],
    "audit_rounds": REPORT["audit_rounds"],
    "audit_violations": REPORT["audit_violations"],
    "t_ret": THR.get("embedding", {}).get("t_ret"),
    "t_strong": THR.get("embedding", {}).get("t_strong"),
    "t_cells": THR["verify"]["t_cells"],
    "h_exact": THR["phash"]["h_exact"],
    "rule_recall_light": round(THR["rule"]["recall_light"], 4),
    "rule_recall_heavy": round(THR["rule"]["recall_heavy"], 4),
    "rule_false_positives": THR["rule"]["false_positives"],
    "data_root_gb": round(used_gb, 2),
    "run_hours": round(RUN.hours(), 2),
}
RUN.save_json(summary, "summary.json")
print(pd.Series(summary).to_string())
st = pd.Series(RUN.stage_times) / 60
RUN.save_csv(st.rename("minutes").to_frame(), "stage_times.csv")
fig, ax = plt.subplots(figsize=(8, 3.5))
st.plot.barh(ax=ax, logx=True)
ax.set_xlabel("minutes (log scale)")
RUN.show(fig, "stage_times")

## 19. Package reports

Zips metrics, plots, logs, code and the manifest into `outputs.zip`. The materialised data in `nss_data/` is left out because it is already part of this notebook's Output and is consumed from there. After **Save Version > Save & Run All**, the zip and all files also appear in the notebook's **Output** tab.

In [ ]:
import shutil
import zipfile

from IPython.display import FileLink, display
from tqdm.auto import tqdm

WORK = C.WORK
ZIP_PATH = WORK / "outputs.zip"
MANIFEST_PATH = DATA_ROOT / "nss_manifest.parquet"
EXCLUDE_PARTS = {
    "__pycache__",
    ".ipynb_checkpoints",
    ".cache",
    "nss_data",
    "tmp",
    "wandb",
}
STORED_SUFFIXES = {
    ".zip",
    ".png",
    ".jpg",
    ".jpeg",
    ".pt",
    ".pth",
    ".bin",
    ".safetensors",
    ".gz",
    ".npz",
    ".onnx",
    ".mp4",
    ".parquet",
}
files = [
    p
    for p in WORK.rglob("*")
    if p.is_file()
    and p != ZIP_PATH
    and (
        p == MANIFEST_PATH or not EXCLUDE_PARTS.intersection(p.relative_to(WORK).parts)
    )
]
total_bytes = sum(p.stat().st_size for p in files)
free_bytes = shutil.disk_usage(WORK).free
assert (
    free_bytes > total_bytes * 1.05
), f"Not enough disk space to zip {total_bytes / 1e9:.2f} GB with {free_bytes / 1e9:.2f} GB free."
with zipfile.ZipFile(ZIP_PATH, "w", allowZip64=True) as archive:
    for p in tqdm(files, desc="Zipping outputs", unit="file"):
        method = (
            zipfile.ZIP_STORED
            if p.suffix.lower() in STORED_SUFFIXES
            else zipfile.ZIP_DEFLATED
        )
        archive.write(p, p.relative_to(WORK), compress_type=method)
print(
    f"Zipped {len(files)} files ({total_bytes / 1e9:.2f} GB) into {ZIP_PATH}",
    flush=True,
)
display(FileLink(os.path.relpath(ZIP_PATH)))